# Chukotka / antimeridian data gap diagnosis

context: i asked an AI agent to try and figure out and explore this data gap issue i found


We see missing data over the Chukotka peninsula (~62-70N, straddling the antimeridian at 180) in the global snowmelt runoff onset dataset. This notebook systematically separates four hypotheses:

| # | Hypothesis | Diagnosed by |
|---|-----------|--------------|
| H1 | **Sentinel-1 availability** — PC's S1 RTC collection is simply sparse/absent in far-east Russia | Section 6: STAC item counts per longitude bin |
| H2 | **MODIS snow mask availability** — the seasonal snow mask (pipeline input) has no data over Chukotka | Section 4/5: inspect legacy + new MODIS stores in native sinusoidal projection |
| H3 | **Pipeline reprojection** — `rio.clip_box(crs=4326)` + `reproject_match` (or `odc.reproject`) of the *sinusoidal* mask onto the S1 lat/lon grid breaks near +/-180 (Chukotka maps to BOTH extreme edges of the sinusoidal x-axis) | Section 4b: replicate the exact pipeline call on real edge tiles, rasterio vs odc |
| H4 | **Visualization / coarsening artifact** — full-res store has the data, the coarsened store or plotting loses it | Sections 2-3: per-longitude valid fraction, full-res vs coarsened |

Section 1 first checks whether the pipeline tiles covering Chukotka were even processed successfully.

**Note:** the v9 production run used `seasonal_snow_mask_reproject_method = rasterio` (see `config/global_config_v9.txt`), i.e. the `rio.clip_box` + `rio.reproject_match` path in `processing.get_spatiotemporal_snow_cover_mask`.

In [ ]:
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
import rasterio
import shapely.geometry
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import odc.geo.xr
from odc.geo.xr import xr_zeros
import pystac_client
import planetary_computer

from global_snowmelt_runoff_onset.config import Config, Tile
from global_snowmelt_runoff_onset.processing import get_spatiotemporal_snow_cover_mask

xr.set_options(keep_attrs=True)

In [ ]:
config = Config('config/global_config_v9.txt')
print('reproject method used in production:', 'rasterio')

In [ ]:
# Optional: spin up a coiled cluster for the heavier full-res cells.
# Everything below also works on the local dask scheduler, just slower.

# import coiled
# cluster = coiled.Cluster(idle_timeout='10 minutes', n_workers=2, worker_memory='256 GB',
#                          spot_policy='spot', workspace='uwtacolab',
#                          environ={'GDAL_DISABLE_READDIR_ON_OPEN': 'EMPTY_DIR'})
# client = cluster.get_client()

## AOI definition

Chukotka straddles the antimeridian, so we define **two** boxes in EPSG:4326 — one just west of 180 (E longitudes) and one just east of it (W longitudes). For continuous plotting we shift the eastern box +360 so the longitude axis runs 165 -> 192 (plotting convenience only, not a CRS operation).

In [ ]:
LAT_MIN, LAT_MAX = 62.0, 70.5
LON_W = (165.0, 179.999)    # west of the antimeridian (positive longitudes)
LON_E = (-179.999, -168.0)  # east of the antimeridian (negative longitudes)
WY_CHECK = 2020             # water year used for single-year MODIS mask plots

box_w = shapely.geometry.box(LON_W[0], LAT_MIN, LON_W[1], LAT_MAX)
box_e = shapely.geometry.box(LON_E[0], LAT_MIN, LON_E[1], LAT_MAX)
aoi_gdf = gpd.GeoDataFrame({'side': ['west_of_antimeridian', 'east_of_antimeridian']},
                           geometry=[box_w, box_e], crs='EPSG:4326')
aoi_gdf

In [ ]:
def robust_slice(ds, dim, vmin, vmax):
    """Slice a coordinate regardless of ascending/descending order."""
    if ds[dim].values[0] > ds[dim].values[-1]:
        return ds.sel({dim: slice(vmax, vmin)})
    return ds.sel({dim: slice(vmin, vmax)})


def sel_chukotka(ds, lat_name='latitude', lon_name='longitude'):
    """Select the Chukotka AOI from a global EPSG:4326 dataset.

    Concatenates the slice west of the antimeridian with the slice east of it
    (shifted +360), giving a continuous 165->192 longitude axis for diagnostics.
    """
    ds_lat = robust_slice(ds, lat_name, LAT_MIN, LAT_MAX)
    w = robust_slice(ds_lat, lon_name, *LON_W)
    e = robust_slice(ds_lat, lon_name, *LON_E)
    e = e.assign_coords({lon_name: e[lon_name] + 360})
    return xr.concat([w, e], dim=lon_name).sortby(lon_name)


def wrap_edges(edges):
    """Map longitudes to the shifted 165..192 frame used for plotting."""
    return [e + 360 if e < 0 else e for e in edges]

## Section 1 — Were the pipeline tiles over Chukotka processed?

Before blaming data or reprojection: do processed (green) tiles even exist on both sides of 180? Failed (red) or never-attempted (orange) tiles here would explain the gap at the tile level — check `error_messages` for the reprojection smoking gun.

In [ ]:
tiles_gdf = config.valid_tiles_gdf.copy()
chuk_tiles_gdf = tiles_gdf[tiles_gdf.intersects(box_w) | tiles_gdf.intersects(box_e)].copy()
chuk_tiles_gdf['minx'] = chuk_tiles_gdf.geometry.bounds['minx']
chuk_tiles_gdf['maxx'] = chuk_tiles_gdf.geometry.bounds['maxx']

show_cols = [c for c in ['row', 'col', 'minx', 'maxx', 'percent_valid_snow_pixels',
                         'success', 'error_messages'] if c in chuk_tiles_gdf.columns]
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 120):
    display(chuk_tiles_gdf[show_cols].sort_values(['row', 'col']))

print(chuk_tiles_gdf['success'].value_counts(dropna=False))

In [ ]:
proj = ccrs.PlateCarree(central_longitude=180)
f, ax = plt.subplots(figsize=(15, 6), subplot_kw=dict(projection=proj))
ax.add_feature(cfeature.LAND, facecolor='0.85', zorder=0)
ax.coastlines(resolution='50m', linewidth=0.5, zorder=1)

status_groups = {
    'processed (success=True)': (chuk_tiles_gdf['success'] == True, 'green'),
    'failed (success=False)': (chuk_tiles_gdf['success'] == False, 'red'),
    'unprocessed (success=NaN)': (chuk_tiles_gdf['success'].isna(), 'orange'),
}
for label, (mask, color) in status_groups.items():
    sub = chuk_tiles_gdf[mask]
    if len(sub):
        ax.add_geometries(sub.geometry, crs=ccrs.PlateCarree(), facecolor=color,
                          edgecolor='black', alpha=0.4, linewidth=0.5)
    ax.plot([], [], 's', color=color, label=f'{label} ({len(sub)})')

for _, t in chuk_tiles_gdf.iterrows():
    cen = t.geometry.centroid
    ax.text(cen.x, cen.y, f'Tile(r={int(t.row)},\nc={int(t.col)})',
            transform=ccrs.PlateCarree(), ha='center', va='center', fontsize=5.5)

ax.add_geometries(aoi_gdf.geometry, crs=ccrs.PlateCarree(), facecolor='none',
                  edgecolor='blue', linewidth=2)
ax.plot([180, 180], [LAT_MIN, LAT_MAX], transform=ccrs.PlateCarree(),
        color='magenta', ls='--', lw=2, label='antimeridian')
ax.set_extent([163, 194, 61, 71.5], crs=ccrs.PlateCarree())
ax.gridlines(draw_labels=True, linewidth=0.3)
ax.legend(loc='lower left')
ax.set_title('Pipeline tile processing status over Chukotka')
plt.show()

## Section 2 — Full-resolution runoff onset dataset (H4 baseline)

Pull the AOI out of the full-res global store and look at (a) the median runoff onset map, (b) the per-block valid-data fraction, and (c) the per-longitude-column valid fraction with tile edges overlaid. A hard cutoff exactly at 180 or at tile edges points at the pipeline; a diffuse fade points at data availability.

In [ ]:
global_runoff_onset_ds = xr.open_zarr(config.global_runoff_store, consolidated=True, decode_coords='all')
global_runoff_onset_ds

In [ ]:
chuk_full_ds = sel_chukotka(global_runoff_onset_ds)
chuk_full_ds

In [ ]:
%%time
disp_factor = 20  # display coarsening only

full_median_disp = (chuk_full_ds['runoff_onset_median']
                    .coarsen(latitude=disp_factor, longitude=disp_factor, boundary='trim')
                    .mean().compute())
full_valid_disp = (chuk_full_ds['runoff_onset_median'].notnull()
                   .coarsen(latitude=disp_factor, longitude=disp_factor, boundary='trim')
                   .mean().compute())

In [ ]:
f, axs = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
full_median_disp.plot.imshow(ax=axs[0], cmap='viridis', vmin=110, vmax=270,
                             cbar_kwargs={'label': 'runoff onset median [DOWY]'})
full_valid_disp.plot.imshow(ax=axs[1], cmap='magma', vmin=0, vmax=1,
                            cbar_kwargs={'label': 'valid-pixel fraction per block'})
axs[0].set_title(f'Full-res runoff_onset_median (display-coarsened x{disp_factor})')
axs[1].set_title('Full-res valid data fraction')
for ax in axs:
    ax.axvline(180, color='red', ls='--', lw=1.5)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
%%time
full_frac = chuk_full_ds['runoff_onset_median'].notnull().mean('latitude').compute()

tile_edges = wrap_edges(sorted({b for g in chuk_tiles_gdf.geometry
                                for b in (g.bounds[0], g.bounds[2])}))

f, ax = plt.subplots(figsize=(16, 4))
full_frac.plot(ax=ax, lw=1.5)
for e in tile_edges:
    ax.axvline(e, color='0.8', lw=0.5, zorder=0)
ax.axvline(180, color='red', ls='--', lw=1.5, label='antimeridian')
ax.set_ylabel('valid fraction of latitude column')
ax.set_title('Full-res runoff_onset_median: valid fraction per longitude (gray = tile edges)')
ax.legend()
plt.show()

In [ ]:
%%time
# Optional (heavier): per-water-year availability, to see whether the gap is
# year-specific (e.g. missing S1 acquisitions in some years) or persistent.
wy_frac = (chuk_full_ds['runoff_onset'].notnull()
           .isel(latitude=slice(None, None, 4))  # thin latitude to lighten the read
           .mean('latitude').compute())

wy_frac.plot.line(x='longitude', col='water_year', col_wrap=5, figsize=(20, 6))
plt.show()

## Section 3 — Coarsened dataset (H4)

Same diagnostics on the factor-20 coarsened store used by the visualization notebooks. **If the full-res store has data where the coarsened store does not, the problem was introduced at the coarsening/visualization stage (H4). If the gaps are identical, the problem is upstream.**

In [ ]:
coarsen_factor = 20
store = config.azure_blob_fs.get_mapper(
    f"snowmelt/snowmelt_runoff_onset/coarsened/global_{config.version}_coarsened_{coarsen_factor}_ds.zarr")
global_coarsened_ds = xr.open_zarr(store, consolidated=True, decode_coords='all', chunks='auto')
global_coarsened_ds = global_coarsened_ds.rio.write_crs('EPSG:4326')
global_coarsened_ds

In [ ]:
chuk_coarse_ds = sel_chukotka(global_coarsened_ds)

coarse_median = chuk_coarse_ds['runoff_onset_median'].compute()
f, axs = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
coarse_median.plot.imshow(ax=axs[0], cmap='viridis', vmin=110, vmax=270,
                          cbar_kwargs={'label': 'runoff onset median [DOWY]'})
coarse_median.notnull().plot.imshow(ax=axs[1], cmap='gray', vmin=0, vmax=1,
                                    cbar_kwargs={'label': 'has data'})
axs[0].set_title('Coarsened (x20) runoff_onset_median')
axs[1].set_title('Coarsened valid data mask')
for ax in axs:
    ax.axvline(180, color='red', ls='--', lw=1.5)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# plot all 10 years of chuk coarsened runoff onset (chuk_coarse_ds)
f, axs = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
for wy, ax in zip(range(2015, 2025), axs.flat):
    coarse_wy = chuk_coarse_ds['runoff_onset'].sel(water_year=wy).compute()
    coarse_wy.plot.imshow(ax=ax, cmap='viridis', vmin=110, vmax=270,
                          cbar_kwargs={'label': 'runoff onset [DOWY]'})
    ax.set_title(f'Water Year {wy}')
    ax.axvline(180, color='red', ls='--', lw=1.5)
    #ax.set_aspect('equal')
plt.tight_layout()

In [ ]:
coarse_frac = chuk_coarse_ds['runoff_onset_median'].notnull().mean('latitude').compute()

f, ax = plt.subplots(figsize=(16, 4))
full_frac.plot(ax=ax, lw=1.5, label='full-res store')
coarse_frac.plot(ax=ax, lw=1.5, label='coarsened store (x20)')
for e in tile_edges:
    ax.axvline(e, color='0.8', lw=0.5, zorder=0)
ax.axvline(180, color='red', ls='--', lw=1.5, label='antimeridian')
ax.set_ylabel('valid fraction of latitude column')
ax.set_title('H4 check: valid fraction per longitude, full-res vs coarsened')
ax.legend()
plt.show()

## Section 4 — Legacy MODIS seasonal snow mask (pipeline input, H2)

`global_modis_snow_cover.zarr` is stored in the **MODIS sinusoidal** projection. Chukotka west of 180 maps to the far-right edge of the sinusoidal x-axis; east of 180 maps to the far-left edge. First: does the store itself have data over Chukotka (H2)?

In [ ]:
legacy_snow_ds = xr.open_zarr(config.snow_phenology_store, consolidated=True, decode_coords='all')
print('CRS:', legacy_snow_ds.rio.crs)
legacy_snow_ds

In [ ]:
%%time
# Clip each side of the antimeridian separately using the same call the
# pipeline makes (rio.clip_box with 4326 bounds against the sinusoidal store).
legacy_clips = {}
for side, box in [('west_of_antimeridian', box_w), ('east_of_antimeridian', box_e)]:
    try:
        legacy_clips[side] = (legacy_snow_ds['max_consec_snow_days']
                              .sel(water_year=WY_CHECK)
                              .rio.clip_box(*box.bounds, crs='EPSG:4326')
                              .compute())
        print(f"{side}: clipped shape {legacy_clips[side].shape}, "
              f"valid frac {float(legacy_clips[side].notnull().mean()):.3f}")
    except Exception as e:
        legacy_clips[side] = None
        print(f"{side}: clip_box FAILED -> {type(e).__name__}: {e}")

In [ ]:
f, axs = plt.subplots(1, 2, figsize=(18, 6))
for ax, (side, da) in zip(axs, legacy_clips.items()):
    if da is not None:
        da.plot.imshow(ax=ax, cmap='Blues', vmin=0, vmax=365)
        ax.set_aspect('equal')
    ax.set_title(f'legacy max_consec_snow_days WY{WY_CHECK}\n{side} (native sinusoidal)')
plt.tight_layout()
plt.show()

In [ ]:
# Reproject each clipped side to EPSG:4326 and compute the per-longitude
# fraction of pixels meeting the seasonal-snow threshold, for the summary plot.
snow_fracs = []
for side, da in legacy_clips.items():
    if da is None:
        continue
    da4326 = da.rio.reproject('EPSG:4326', resampling=rasterio.enums.Resampling.bilinear)
    da4326 = da4326.where(da4326 != da4326.rio.nodata) if da4326.rio.nodata is not None else da4326
    frac = (da4326 >= config.min_consec_snow_days_for_seasonal_snow).mean('y')
    frac = frac.assign_coords(x=wrap_edges(frac.x.values))
    snow_fracs.append(frac)
legacy_snow_frac = xr.concat(snow_fracs, dim='x').sortby('x') if snow_fracs else None

if legacy_snow_frac is not None:
    f, ax = plt.subplots(figsize=(16, 4))
    legacy_snow_frac.plot(ax=ax)
    ax.axvline(180, color='red', ls='--', lw=1.5)
    ax.set_title(f'Legacy MODIS mask: fraction of latitude column with seasonal snow '
                 f'(>= {config.min_consec_snow_days_for_seasonal_snow} consec days), WY{WY_CHECK}')
    plt.show()

### Section 4b — Replicate the pipeline reprojection on real edge tiles (H3)

This is the direct test of the reprojection hypothesis: for actual v9 tiles touching the antimeridian (from both sides), build the same lat/lon target grid the pipeline loads S1 into (`tile.geobox`) and run `get_spatiotemporal_snow_cover_mask` with **both** `rasterio` and `odc` methods. If the native sinusoidal store has snow here (Section 4) but these masks come back empty/garbled or the call fails, the pipeline reprojection is the culprit — and comparing the two methods shows whether switching fixes it.

In [ ]:
edge_tiles_gdf = chuk_tiles_gdf[(chuk_tiles_gdf['maxx'] > 179.5) | (chuk_tiles_gdf['minx'] < -179.5)]
# also grab a control tile well away from the antimeridian (~170E)
control_gdf = chuk_tiles_gdf[(chuk_tiles_gdf['minx'] > 168) & (chuk_tiles_gdf['maxx'] < 173)]
# and the exact pair of tiles flanking the antimeridian itself -- see Section 10 for why they matter
antimeridian_pair_gdf = chuk_tiles_gdf[(chuk_tiles_gdf['row'] == 8) & (chuk_tiles_gdf['col'].isin([0, 243]))]
test_tiles_gdf = pd.concat([edge_tiles_gdf.head(3), control_gdf.head(1), antimeridian_pair_gdf])
test_tiles_gdf = test_tiles_gdf.drop_duplicates(subset=['row', 'col'])
test_tiles_gdf[['row', 'col', 'minx', 'maxx'] + (['success'] if 'success' in test_tiles_gdf else [])]

In [ ]:
def fake_s1_grid(tile):
    """Emulate the S1 target grid the pipeline loads for a tile."""
    da = xr_zeros(tile.geobox, dtype='float32')
    da = da.expand_dims(time=[np.datetime64('2020-06-01', 'ns')])
    return da.to_dataset(name='vv')


mask_results = {}
for _, trow in test_tiles_gdf.iterrows():
    tile = config.get_tile(int(trow.row), int(trow.col))
    s1_like_ds = fake_s1_grid(tile)
    for method in ['rasterio', 'odc']:
        key = (tile.index, method)
        try:
            mask_ds = get_spatiotemporal_snow_cover_mask(
                s1_like_ds, tile.bbox_gdf, config.snow_phenology_store,
                extend_search_window_beyond_SDD_days=config.extend_search_window_beyond_SDD_days,
                min_consec_snow_days_for_seasonal_snow=config.min_consec_snow_days_for_seasonal_snow,
                reproject_method=method)
            snow = mask_ds['binary_seasonal_snow_cover_presence'].sel(water_year=WY_CHECK).compute()
            mask_results[key] = snow
            print(f"tile {tile.index} [{method:8s}]  seasonal-snow fraction = {float(snow.mean()):.4f}")
        except Exception as e:
            mask_results[key] = None
            print(f"tile {tile.index} [{method:8s}]  FAILED -> {type(e).__name__}: {e}")

In [ ]:
import contextily as ctx

_context_img_cache = {}


def tile_context_image(tk, bounds):
    """Esri World Imagery for a tile's extent, warped to EPSG:4326 (cached)."""
    if tk not in _context_img_cache:
        minx, miny, maxx, maxy = bounds
        img, ext = ctx.bounds2img(minx, miny, maxx, maxy, ll=True,
                                  source=ctx.providers.Esri.WorldImagery, zoom=9)
        img, ext = ctx.warp_tiles(img, ext, 'EPSG:4326')
        _context_img_cache[tk] = (img, ext)
    return _context_img_cache[tk]


def plot_mask_comparison(results, mask_label):
    """Per-tile 4-column comparison:
    context imagery | coarsened runoff onset (WY_CHECK) | rasterio mask | odc mask."""
    tile_keys = list(dict.fromkeys(k[0] for k in results))
    f, axs = plt.subplots(len(tile_keys), 4, figsize=(24, 5 * len(tile_keys)), squeeze=False)
    for i, tk in enumerate(tile_keys):
        bounds = config.get_tile(*tk).bbox_gdf.total_bounds
        minx, miny, maxx, maxy = bounds

        ax = axs[i, 0]
        try:
            img, ext = tile_context_image(tk, bounds)
            ax.imshow(img, extent=ext)
            ax.set_xlim(minx, maxx)
            ax.set_ylim(miny, maxy)
            ax.set_aspect('equal')
        except Exception as e:
            ax.text(0.5, 0.5, f'basemap failed:\n{type(e).__name__}', ha='center',
                    va='center', transform=ax.transAxes, color='red')
        ax.set_title(f'tile {tk} — context (Esri World Imagery)')

        ax = axs[i, 1]
        coarse_tile = robust_slice(
            robust_slice(global_coarsened_ds['runoff_onset'].sel(water_year=WY_CHECK),
                         'latitude', miny, maxy),
            'longitude', minx, maxx).compute()
        if coarse_tile.size:
            coarse_tile.plot.imshow(ax=ax, cmap='viridis', vmin=110, vmax=270, add_colorbar=False)
            ax.set_aspect('equal')
        else:
            ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                    transform=ax.transAxes, color='red')
        ax.set_title(f'tile {tk} — coarsened runoff_onset WY{WY_CHECK}')

        for j, method in enumerate(['rasterio', 'odc']):
            ax = axs[i, j + 2]
            snow = results.get((tk, method))
            if snow is not None:
                snow.plot.imshow(ax=ax, cmap='gray_r', vmin=0, vmax=1, add_colorbar=False)
                ax.set_aspect('equal')
            else:
                ax.text(0.5, 0.5, 'FAILED', ha='center', va='center',
                        transform=ax.transAxes, color='red')
            ax.set_title(f'tile {tk} — {method}\n{mask_label} seasonal snow WY{WY_CHECK}')
    plt.tight_layout()
    plt.show()


plot_mask_comparison(mask_results, 'legacy mask')

## Section 5 — New MODIS_snow_phenology dataset (Icechunk / Zarr v3)

Same Chukotka check on the new store, both to validate the new dataset near the antimeridian and to cross-check the legacy store (if the new one has data where the legacy one doesn't, the legacy mask inherited a gap).

In [ ]:
sys.path.insert(0, '/home/eric/repos/MODIS_snow_phenology')
from modis_snow_phenology.config import Config as PhenologyConfig

phenology_config = PhenologyConfig('config/config_with_secrets_v1.txt')
phenology_session = phenology_config.open_icechunk_repo().readonly_session('main')
phenology_ds = xr.open_zarr(phenology_session.store, zarr_format=3, consolidated=False,
                            decode_coords='all')
print('CRS:', phenology_ds.rio.crs)
phenology_ds

In [ ]:
%%time
phenology_clips = {}
for side, box in [('west_of_antimeridian', box_w), ('east_of_antimeridian', box_e)]:
    try:
        phenology_clips[side] = (phenology_ds['max_consec_snow_days']
                                 .sel(water_year=WY_CHECK)
                                 .rio.clip_box(*box.bounds, crs='EPSG:4326')
                                 .compute())
        print(f"{side}: clipped shape {phenology_clips[side].shape}, "
              f"valid frac {float(phenology_clips[side].notnull().mean()):.3f}")
    except Exception as e:
        phenology_clips[side] = None
        print(f"{side}: clip_box FAILED -> {type(e).__name__}: {e}")

In [ ]:
f, axs = plt.subplots(2, 2, figsize=(18, 11))
for j, side in enumerate(['west_of_antimeridian', 'east_of_antimeridian']):
    for i, (label, clips) in enumerate([('legacy (Zarr v2)', legacy_clips),
                                        ('new phenology (Icechunk)', phenology_clips)]):
        ax = axs[i, j]
        da = clips.get(side)
        if da is not None:
            da.plot.imshow(ax=ax, cmap='Blues', vmin=0, vmax=365)
            ax.set_aspect('equal')
        else:
            ax.text(0.5, 0.5, 'clip FAILED / empty', ha='center', va='center',
                    transform=ax.transAxes, color='red')
        ax.set_title(f'{label} — {side}\nmax_consec_snow_days WY{WY_CHECK}')
plt.tight_layout()
plt.show()

### Section 5b — Pipeline reprojection replication with the NEW phenology store (H3)

Same 3-column comparison as Section 4b (coarsened runoff onset | `rasterio` | `odc`), but with the seasonal-snow mask derived from the new Icechunk store. Note: the pipeline's `get_spatiotemporal_snow_cover_mask` cannot open the Icechunk store directly (it calls `xr.open_zarr(store, consolidated=True)` internally), so this cell replicates its clip + reproject logic on the already-opened `phenology_ds` — meaning that function will need a tweak before the pipeline can ever be re-run against the new store.

In [ ]:
%%time
def replicate_mask_reprojection(mask_ds, tile, s1_like_ds, method, wy=WY_CHECK):
    """Mirror the clip + reproject step of processing.get_spatiotemporal_snow_cover_mask
    for an already-opened seasonal snow mask dataset (single water year)."""
    clip_ds = mask_ds[['max_consec_snow_days']].sel(water_year=wy).rio.clip_box(
        *tile.bbox_gdf.total_bounds, crs='EPSG:4326')
    if method == 'rasterio':
        out = clip_ds.rio.reproject_match(
            s1_like_ds.isel(time=0),
            resampling=rasterio.enums.Resampling.bilinear).rename({'x': 'longitude', 'y': 'latitude'})
    elif method == 'odc':
        out = clip_ds.odc.reproject(s1_like_ds.odc.geobox, resampling='bilinear')
    return (out['max_consec_snow_days'] >= config.min_consec_snow_days_for_seasonal_snow).compute()


def compute_phenology_mask_results(wy):
    """Run replicate_mask_reprojection (rasterio + odc) over test_tiles_gdf for one water year."""
    results = {}
    for _, trow in test_tiles_gdf.iterrows():
        tile = config.get_tile(int(trow.row), int(trow.col))
        s1_like_ds = fake_s1_grid(tile)
        for method in ['rasterio', 'odc']:
            key = (tile.index, method)
            try:
                snow = replicate_mask_reprojection(phenology_ds, tile, s1_like_ds, method, wy=wy)
                results[key] = snow
                print(f"WY{wy} tile {tile.index} [{method:8s}]  seasonal-snow fraction = {float(snow.mean()):.4f}")
            except Exception as e:
                results[key] = None
                print(f"WY{wy} tile {tile.index} [{method:8s}]  FAILED -> {type(e).__name__}: {e}")
    return results


phenology_mask_results = compute_phenology_mask_results(WY_CHECK)

In [ ]:
plot_mask_comparison(phenology_mask_results, 'new phenology mask')

## Section 6 — Sentinel-1 RTC availability (H1)

Count Planetary Computer `sentinel-1-rtc` items in 1-degree longitude bins across Chukotka for the full study period. A collapse in counts approaching/east of 180 means the input SAR data simply isn't there (a known weak spot of the PC RTC collection in far-east Russia).

In [ ]:
%%time
import time
from pystac_client.exceptions import APIError

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace)

# The full study period is the most definitive but the heaviest for the API.
# If PC keeps returning 504s, shorten to a single melt season -- the *spatial*
# pattern across the antimeridian is what matters for H1:
# S1_DATETIME = '2020-03-01/2020-08-31'
S1_DATETIME = f'{config.start_date}/{config.end_date}'


def count_s1_items(bbox, datetime_range, tries=4):
    """Count matching S1 RTC items, retrying on PC gateway timeouts (504s)."""
    for attempt in range(tries):
        try:
            search = catalog.search(collections=['sentinel-1-rtc'], bbox=bbox,
                                    datetime=datetime_range, limit=1000)
            n = search.matched()
            if n is None:  # PC often omits numberMatched; count by paging
                n = sum(1 for _ in search.items_as_dicts())
            return float(n)
        except APIError:
            if attempt == tries - 1:
                print('    still failing, giving up on this bin -> NaN')
                return np.nan
            wait = 5 * 2 ** attempt
            print(f'    APIError from PC STAC, retry {attempt + 1}/{tries - 1} in {wait}s')
            time.sleep(wait)


def lon_bin_bbox(lon0):
    """1-degree bbox for a bin in the shifted 165..192 frame."""
    a = lon0 if lon0 < 180 else lon0 - 360
    b = lon0 + 1 if lon0 + 1 <= 180 else lon0 + 1 - 360
    return [a, 64.0, b, 68.0]


lon_bins = np.arange(165.0, 192.0, 1.0)  # shifted frame; >=180 means negative lons
s1_counts = []
for lon0 in lon_bins:
    n = count_s1_items(lon_bin_bbox(lon0), S1_DATETIME)
    s1_counts.append(n)
    print(f'lon bin [{lon0:6.1f}, {lon0 + 1:6.1f}):  {n:7.0f} items')

s1_counts = np.array(s1_counts)

In [ ]:
f, ax = plt.subplots(figsize=(16, 4))
ax.bar(lon_bins + 0.5, s1_counts, width=0.9)
ax.axvline(180, color='red', ls='--', lw=1.5, label='antimeridian')
ax.set_xlabel('longitude (shifted frame, >180 = negative lons)')
ax.set_ylabel(f'S1 RTC items {config.start_date} to {config.end_date}\n(lat 64-68)')
ax.set_title('H1 check: Sentinel-1 RTC availability across Chukotka')
ax.legend()
plt.show()

### Per-water-year scene count heatmaps

One spatial (lon x lat) heatmap of S1 RTC scene counts for EACH water year (Oct 1 -> Sep 30). Rather than one STAC count query per (year, lat, lon) bin (~2000 requests), this fetches the actual item footprints once per water year (2 searches per WY, one per side of the antimeridian) and bins them onto a 0.5-degree grid locally. Footprints east of 180 are shifted into the continuous 165->192 longitude frame; scenes straddling 180 are deduplicated by item id.

In [ ]:
%%time
def to_lon360(geom):
    """Shift a footprint's negative longitudes +360 (continuous 165->192 frame)."""
    return shapely.transform(
        geom, lambda c: np.column_stack((np.where(c[:, 0] < 0, c[:, 0] + 360, c[:, 0]), c[:, 1])))


def _sanitize_geometry(geom):
    """Drop degenerate empty polygon parts that occasionally show up in antimeridian-adjacent STAC
    footprints -- e.g. S1A_EW_GRDM_1SDH_20180401T181439_20180401T181539_021277_02499C_rtc ships a
    MultiPolygon with an empty [] entry, which crashes shapely's shape() inside
    GeoDataFrame.from_features. This is a metadata quirk in the upstream antimeridian-splitting
    logic, not a real second part of the scene."""
    if geom is not None and geom.get('type') == 'MultiPolygon':
        coords = [poly for poly in geom['coordinates'] if poly and poly[0]]
        if not coords:
            return None
        geom = {**geom, 'coordinates': coords}
    return geom


def fetch_s1_items_gdf(bbox, datetime_range, collection='sentinel-1-rtc', tries=4):
    """Fetch S1 item footprints as a GeoDataFrame, retrying on PC 504s."""
    feats = None
    for attempt in range(tries):
        try:
            search = catalog.search(collections=[collection], bbox=bbox,
                                    datetime=datetime_range, limit=1000)
            feats = list(search.items_as_dicts())
            break
        except APIError:
            if attempt == tries - 1:
                print(f'    bbox {bbox}: still failing, returning empty')
                break
            wait = 5 * 2 ** attempt
            print(f'    APIError from PC STAC, retry {attempt + 1}/{tries - 1} in {wait}s')
            time.sleep(wait)
    if not feats:
        return gpd.GeoDataFrame({'id': []}, geometry=[], crs='EPSG:4326')
    for f in feats:
        f['geometry'] = _sanitize_geometry(f['geometry'])
    n_dropped = sum(f['geometry'] is None for f in feats)
    if n_dropped:
        print(f'    dropped {n_dropped} item(s) with degenerate (empty-ring) footprints')
    feats = [f for f in feats if f['geometry'] is not None]
    gdf = gpd.GeoDataFrame.from_features(feats, crs='EPSG:4326')
    gdf['id'] = [f['id'] for f in feats]
    gdf.geometry = gdf.geometry.apply(to_lon360)
    return gdf


s1_items_by_wy = {}
for wy in config.water_years:
    wy_datetime = f'{wy - 1}-10-01/{wy}-09-30'
    parts = [fetch_s1_items_gdf([LON_W[0], LAT_MIN, LON_W[1], LAT_MAX], wy_datetime),
             fetch_s1_items_gdf([LON_E[0], LAT_MIN, LON_E[1], LAT_MAX], wy_datetime)]
    gdf = pd.concat(parts, ignore_index=True)
    gdf = gdf.drop_duplicates(subset='id')  # scenes straddling 180 hit both searches
    s1_items_by_wy[wy] = gdf
    print(f'WY{wy}: {len(gdf):5d} unique S1 RTC scenes')

In [ ]:
GRID_RES = 0.05
glons = np.arange(165.0, 192.0 + GRID_RES, GRID_RES)
glats = np.arange(np.floor(LAT_MIN), np.ceil(LAT_MAX) + GRID_RES, GRID_RES)

grid_gdf = gpd.GeoDataFrame(
    {'cell': range((len(glats) - 1) * (len(glons) - 1))},
    geometry=[shapely.geometry.box(x, y, x + GRID_RES, y + GRID_RES)
              for y in glats[:-1] for x in glons[:-1]],
    crs='EPSG:4326')

count_grids = {}
for wy, gdf in s1_items_by_wy.items():
    counts = np.zeros(len(grid_gdf))
    if len(gdf):
        joined = gpd.sjoin(grid_gdf, gdf[['geometry']], how='inner', predicate='intersects')
        vc = joined.groupby('cell').size()
        counts[vc.index] = vc.values
    count_grids[wy] = counts.reshape(len(glats) - 1, len(glons) - 1)

vmax = max(g.max() for g in count_grids.values())

f, axs = plt.subplots(2, 5, figsize=(24, 8), sharex=True, sharey=True)
for wy, ax in zip(config.water_years, axs.flat):
    mesh = ax.pcolormesh(glons, glats, count_grids[wy], cmap='magma', vmin=0, vmax=vmax)
    ax.axvline(180, color='red', ls='--', lw=1)
    ax.set_title(f'WY{wy} ({len(s1_items_by_wy[wy])} scenes)')
    #ax.set_aspect('equal')
f.suptitle('S1 RTC scene counts per water year (0.5-degree bins, footprint intersections)', y=1.02)
f.colorbar(mesh, ax=axs, label='S1 RTC scenes', shrink=0.8)
plt.show()

## Section 7 — Cross-dataset summary

Everything on one longitude axis: where exactly does each dataset lose data relative to the antimeridian, the tile edges, and the S1 availability?

In [ ]:
f, ax = plt.subplots(figsize=(17, 6))
full_frac.plot(ax=ax, lw=2, label='runoff onset — full-res store')
coarse_frac.plot(ax=ax, lw=2, label='runoff onset — coarsened store')
if legacy_snow_frac is not None:
    legacy_snow_frac.plot(ax=ax, lw=1.5, ls=':', label=f'legacy MODIS seasonal-snow frac (WY{WY_CHECK})')
for e in tile_edges:
    ax.axvline(e, color='0.85', lw=0.5, zorder=0)
ax.axvline(180, color='red', ls='--', lw=1.5, label='antimeridian')

ax2 = ax.twinx()
ax2.step(lon_bins + 0.5, s1_counts, where='mid', color='purple', alpha=0.5, label='S1 RTC item count')
ax2.set_ylabel('S1 RTC items (purple)', color='purple')

ax.set_ylabel('valid / snow fraction of latitude column')
ax.set_title('Chukotka antimeridian summary — all datasets')
ax.legend(loc='upper left')
plt.show()

## Section 8 — Combined per-tile view: new mask reprojection vs actual S1 VV availability

Everything relevant to each test tile in one figure. Rows = tiles, columns =

1. context imagery
2. coarsened `runoff_onset` (WY_CHECK)
3. `rasterio`-reprojected new phenology mask
4. `odc`-reprojected new phenology mask
5. **S1 VV valid-observation count for WY_CHECK** — actual Sentinel-1 RTC loaded through the pipeline's own `get_sentinel1_rtc`, on a coarsened copy of the tile geobox (per-pixel count of non-nodata acquisitions)

Column 5 is the ground truth for H1 at tile scale: if the mask columns look fine but column 5 is ~zero, the S1 input itself is missing; if column 5 shows plenty of observations but the runoff onset (column 2) is empty, the problem is in the pipeline (columns 3-4 show whether the mask reprojection is the culprit).

In [ ]:
%%time
from global_snowmelt_runoff_onset.processing import get_sentinel1_rtc

S1_COUNT_ZOOM_OUT = 16  # count on a ~16x coarser grid than the tile: availability-scale, much faster


def compute_s1_vv_counts(wy):
    """Per-pixel count of valid (non-nodata) S1 VV acquisitions for one water year, over test_tiles_gdf."""
    counts = {}
    for _, trow in test_tiles_gdf.iterrows():
        tile = config.get_tile(int(trow.row), int(trow.col))
        count_geobox = tile.geobox.zoom_out(S1_COUNT_ZOOM_OUT)
        try:
            s1_ds = get_sentinel1_rtc(count_geobox, bands=['vv'],
                                      start_date=f'{wy - 1}-10-01',
                                      end_date=f'{wy}-09-30',
                                      chunks_read={'x': 256, 'y': 256, 'time': 1},
                                      fail_on_error=False)
            vv = s1_ds['vv']
            count = (np.isfinite(vv) & (vv != -32768)).sum('time').compute()
            counts[tile.index] = count
            print(f'WY{wy} tile {tile.index}: {s1_ds.sizes["time"]} acquisitions, '
                  f'per-pixel valid count max = {int(count.max())}')
        except Exception as e:
            counts[tile.index] = None
            print(f'WY{wy} tile {tile.index}: FAILED -> {type(e).__name__}: {e}')
    return counts


s1_vv_counts = compute_s1_vv_counts(WY_CHECK)

In [ ]:
def plot_combined_tile_view(mask_results, vv_counts, wy):
    """Section 8 combined per-tile figure (context | coarsened runoff | rasterio/odc mask | S1 VV count)
    for an arbitrary water year."""
    tile_keys = list(dict.fromkeys(k[0] for k in mask_results))
    n_tiles = len(tile_keys)
    col_labels = ['context\n(Esri World Imagery)',
                  f'coarsened runoff_onset\nWY{wy}',
                  'new phenology mask\n(rasterio reproject)',
                  'new phenology mask\n(odc reproject)',
                  f'S1 VV valid obs count\nWY{wy}']

    f, axs = plt.subplots(n_tiles, 5, figsize=(30, 5.5 * n_tiles), squeeze=False)
    for i, tk in enumerate(tile_keys):
        minx, miny, maxx, maxy = config.get_tile(*tk).bbox_gdf.total_bounds

        ax = axs[i, 0]
        try:
            img, ext = tile_context_image(tk, (minx, miny, maxx, maxy))
            ax.imshow(img, extent=ext)
            ax.set_xlim(minx, maxx)
            ax.set_ylim(miny, maxy)
            ax.set_aspect('equal')
        except Exception as e:
            ax.text(0.5, 0.5, f'basemap failed:\n{type(e).__name__}', ha='center',
                    va='center', transform=ax.transAxes, color='red')

        ax = axs[i, 1]
        coarse_tile = robust_slice(
            robust_slice(global_coarsened_ds['runoff_onset'].sel(water_year=wy),
                         'latitude', miny, maxy),
            'longitude', minx, maxx).compute()
        if coarse_tile.size:
            coarse_tile.plot.imshow(ax=ax, cmap='viridis', vmin=110, vmax=270,
                                    cbar_kwargs={'label': 'DOWY', 'shrink': 0.8})
            ax.set_aspect('equal')
        else:
            ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                    transform=ax.transAxes, color='red')

        for k, method in enumerate(['rasterio', 'odc']):
            ax = axs[i, 2 + k]
            snow = mask_results.get((tk, method))
            if snow is not None:
                snow.plot.imshow(ax=ax, cmap='gray_r', vmin=0, vmax=1, add_colorbar=False)
                ax.set_aspect('equal')
            else:
                ax.text(0.5, 0.5, 'FAILED', ha='center', va='center',
                        transform=ax.transAxes, color='red')

        ax = axs[i, 4]
        count = vv_counts.get(tk)
        if count is not None:
            count.plot.imshow(ax=ax, cmap='cividis', vmin=0,
                              cbar_kwargs={'label': 'valid obs', 'shrink': 0.8})
            ax.set_aspect('equal')
        else:
            ax.text(0.5, 0.5, 'S1 load FAILED\n(or no scenes)', ha='center', va='center',
                    transform=ax.transAxes, color='red')

    for j, label in enumerate(col_labels):
        for i in range(n_tiles):
            axs[i, j].set_title(label if i == 0 else '')
        for i, tk in enumerate(tile_keys):
            axs[i, 0].set_ylabel(f'tile {tk}', fontsize=13)
    f.suptitle(f'Section 8 combined per-tile view -- WY{wy}', y=1.005)
    plt.tight_layout()
    plt.show()


plot_combined_tile_view(phenology_mask_results, s1_vv_counts, WY_CHECK)

In [ ]:
WY_2016 = 2016
phenology_mask_results_2016 = compute_phenology_mask_results(WY_2016)
s1_vv_counts_2016 = compute_s1_vv_counts(WY_2016)
plot_combined_tile_view(phenology_mask_results_2016, s1_vv_counts_2016, WY_2016)

In [ ]:
WY_2024 = 2024
phenology_mask_results_2024 = compute_phenology_mask_results(WY_2024)
s1_vv_counts_2024 = compute_s1_vv_counts(WY_2024)
plot_combined_tile_view(phenology_mask_results_2024, s1_vv_counts_2024, WY_2024)

## Section 9 -- Does Sentinel-1 acquisition mode (IW vs EW) change over Chukotka?

Every S1 GRD/RTC STAC item carries a `sar:instrument_mode` property (`IW`, `EW`, `SM`, or `WV`) from the [SAR STAC extension](https://github.com/stac-extensions/sar). Two collection-level facts turn out to matter more than any tile-level reprojection bug:

- **`sentinel-1-rtc`** -- the collection the pipeline's `get_sentinel1_rtc` actually reads -- only ever carries `IW`-mode products. Confirmed below directly against the Planetary Computer collection summary: RTC processing on PC is IW-only, full stop.
- **`sentinel-1-grd`** -- the raw (pre-RTC) collection -- carries `IW`, `EW`, *and* `SM` over Chukotka. ESA's background mission plan tasks this part of the high Arctic in `EW` a meaningful fraction of the time.

If both of those hold, **any scene acquired in EW mode over Chukotka is invisible to the pipeline no matter what the reprojection code does** -- a mode gap sitting upstream of H1-H4 entirely, and orthogonal to the antimeridian itself. The open question is whether that gap is a one-time historical cutover (IW until some date, then EW after -- what prompted this section) or something that cycles (e.g. seasonally, as part of ESA's rolling acquisition plan). We check the full STAC record first (authoritative, goes back to 2015), then cross-check against ESA's own acquisition-plan KML files (only a rolling near-term window, but a useful independent, ESA-native confirmation of "what mode is Chukotka in right now").

In [ ]:
# Collection-level check: does sentinel-1-rtc ever carry anything but IW?
rtc_collection = catalog.get_collection('sentinel-1-rtc')
grd_collection = catalog.get_collection('sentinel-1-grd')
print("sentinel-1-rtc  sar:instrument_mode summary:", rtc_collection.summaries.get_list('sar:instrument_mode'))
print("sentinel-1-grd  sar:instrument_mode summary:", grd_collection.summaries.get_list('sar:instrument_mode'))

# Sanity-check against the RTC scenes already fetched per water year over Chukotka in Section 6
# (s1_items_by_wy) -- these should come back 100% IW if the collection-level summary holds locally too.
for wy, gdf in s1_items_by_wy.items():
    print(f"WY{wy}: RTC scenes over Chukotka by mode -> {gdf['sar:instrument_mode'].value_counts().to_dict()}")

In [ ]:
%%time
grd_items_by_wy = {}
for wy in config.water_years:
    wy_datetime = f'{wy - 1}-10-01/{wy}-09-30'
    parts = [fetch_s1_items_gdf([LON_W[0], LAT_MIN, LON_W[1], LAT_MAX], wy_datetime, collection='sentinel-1-grd'),
             fetch_s1_items_gdf([LON_E[0], LAT_MIN, LON_E[1], LAT_MAX], wy_datetime, collection='sentinel-1-grd')]
    gdf = pd.concat(parts, ignore_index=True).drop_duplicates(subset='id')
    grd_items_by_wy[wy] = gdf
    mode_counts = gdf['sar:instrument_mode'].value_counts().to_dict()
    print(f'WY{wy}: {len(gdf):5d} unique S1 GRD scenes -> {mode_counts}')

In [ ]:
modes = ['IW', 'EW', 'SM']
grd_mode_counts = pd.DataFrame(
    {wy: gdf['sar:instrument_mode'].value_counts().reindex(modes, fill_value=0)
     for wy, gdf in grd_items_by_wy.items()}).T
rtc_counts = pd.Series({wy: len(gdf) for wy, gdf in s1_items_by_wy.items()}, name='RTC scenes (IW only)')

f, ax = plt.subplots(figsize=(14, 5))
grd_mode_counts.plot(kind='bar', stacked=True, ax=ax,
                     color={'IW': 'tab:blue', 'EW': 'tab:orange', 'SM': 'tab:green'})
ax.plot(range(len(rtc_counts)), rtc_counts.reindex(grd_mode_counts.index).values,
        'k--o', label='RTC scenes actually usable by the pipeline (IW only)')
ax.set_ylabel('S1 scenes over Chukotka per water year')
ax.set_xlabel('water year')
ax.set_title('GRD scenes by acquisition mode vs. RTC (IW-only) scenes the pipeline can see')
ax.legend()
plt.show()

ew_fraction = (grd_mode_counts['EW'] / grd_mode_counts[modes].sum(axis=1)).rename('EW fraction of GRD scenes')
print(ew_fraction)

In [ ]:
grd_all = pd.concat(grd_items_by_wy.values(), ignore_index=True).drop_duplicates(subset='id')
grd_all['month'] = pd.to_datetime(grd_all['datetime'], format='ISO8601').dt.tz_convert(None).dt.to_period('M')
monthly_mode = grd_all.groupby(['month', 'sar:instrument_mode']).size().unstack(fill_value=0)
monthly_mode = monthly_mode.reindex(columns=modes, fill_value=0)

f, ax = plt.subplots(figsize=(18, 5))
monthly_mode.plot(ax=ax, color={'IW': 'tab:blue', 'EW': 'tab:orange', 'SM': 'tab:green'})
ax.set_ylabel('S1 GRD scenes per month over Chukotka')
ax.set_title('Monthly IW vs EW acquisition counts, full record -- cyclical, not a one-time cutover')
plt.show()

**What the STAC record actually shows:** EW mode is not a one-time historical cutover -- it is present in every water year 2015-2024, and the monthly breakdown above shows it cycling *seasonally*: EW share rises each fall/winter (roughly Oct-Apr) and drops each late spring/summer, essentially every year of the record. The west-of-antimeridian and east-of-antimeridian sides show almost the same EW fraction, so this is not an antimeridian-specific artifact -- it is a general feature of ESA's background acquisition plan at this latitude. Two practical implications for this pipeline:

- The EW-heavy months (accumulation season) matter for anything that needs winter S1 density, but the melt-season months that actually drive runoff-onset detection are consistently the *most* IW-heavy months of the year -- so this mode gap is a real, upstream data limitation (a refinement of H1) but likely a smaller contributor to the runoff-onset gap than a full-blown seasonal EW cutover would be.
- The steep drop in total scene counts from 2022 onward lines up with the loss of Sentinel-1B in December 2021 (single-satellite constellation until Sentinel-1C came online) -- a separate, well-documented availability dip worth keeping in mind when comparing pre/post-2022 water years anywhere in this dataset, not just Chukotka.

### Bonus cross-check -- ESA's own acquisition-plan KML files

ESA publishes rolling Sentinel-1 mission-planning KML files (per satellite, ~20-day windows, color-coded / tagged by mode) at [sentinels.copernicus.eu/copernicus/sentinel-1/acquisition-plans](https://sentinels.copernicus.eu/copernicus/sentinel-1/acquisition-plans). These only cover the *current* rolling window (the linked "observation scenario archive" only has JPEG images of past cycles, not machine-readable KML), so they cannot replace the STAC-based history above -- but they are a useful independent, ESA-native confirmation of what mode is currently planned over Chukotka, straight from the mission planners rather than derived from processed STAC metadata. Wrapped in a try/except since it depends on scraping a live, occasionally-changing webpage.

In [ ]:
import re
import requests
import xml.etree.ElementTree as ET
from datetime import datetime, timezone

PLANS_URL = 'https://sentinels.copernicus.eu/copernicus/sentinel-1/acquisition-plans'
KML_NS = 'http://www.opengis.net/kml/2.2'


def find_current_plan_hrefs():
    """Scrape the acquisition-plans page for KML links whose [start, end) window brackets now."""
    html = requests.get(PLANS_URL, timeout=30).text
    now = datetime.now(timezone.utc)
    hrefs = {}
    for m in re.finditer(
            r'href="(/documents/d/sentinel/(s1[a-d])_mp_user_(\d{8}t\d{6})_(\d{8}t\d{6}))"', html):
        href, sat, start_s, end_s = m.groups()
        start = datetime.strptime(start_s, '%Y%m%dt%H%M%S').replace(tzinfo=timezone.utc)
        end = datetime.strptime(end_s, '%Y%m%dt%H%M%S').replace(tzinfo=timezone.utc)
        if start <= now <= end:
            hrefs[sat] = href  # last match per satellite wins (most specific/most recent posting)
    return hrefs


def parse_plan_kml(content):
    """Pull Placemark ExtendedData + footprint geometry out of an ESA acquisition-plan KML."""
    root = ET.fromstring(content)
    records = []
    for pm in root.iter(f'{{{KML_NS}}}Placemark'):
        data = {d.get('name'): d.findtext(f'{{{KML_NS}}}value') for d in pm.iter(f'{{{KML_NS}}}Data')}
        coords_el = pm.find(f'.//{{{KML_NS}}}coordinates')
        if coords_el is None or not coords_el.text:
            continue
        coords = [tuple(map(float, c.split(',')))[:2] for c in coords_el.text.split()]
        if len(coords) < 2:
            continue
        data['geometry'] = (shapely.geometry.LineString(coords) if len(coords) == 2
                            else shapely.geometry.Polygon(coords))
        records.append(data)
    return gpd.GeoDataFrame(records, crs='EPSG:4326') if records else None


try:
    plan_hrefs = find_current_plan_hrefs()
    print('current acquisition-plan files:', plan_hrefs)
    plan_gdfs = []
    for sat, href in plan_hrefs.items():
        r = requests.get(f'https://sentinels.copernicus.eu{href}', timeout=30)
        r.raise_for_status()
        gdf = parse_plan_kml(r.content)
        if gdf is not None:
            plan_gdfs.append(gdf)
    plan_gdf = pd.concat(plan_gdfs, ignore_index=True) if plan_gdfs else None
except Exception as e:
    plan_gdf = None
    print(f'ESA acquisition-plan fetch FAILED -> {type(e).__name__}: {e}')

# Note: some datatake footprints straddle the dateline and are stored with unwrapped (e.g. <-180)
# longitudes in the raw KML, so a handful of intersects() checks below near +/-180 may be unreliable --
# this cell is a bonus cross-check, not the primary evidence (that's the STAC analysis above).
if plan_gdf is not None:
    print('planned datatakes by mode, current window:', plan_gdf['Mode'].value_counts().to_dict())
    mode_colors = {'IW': 'tab:blue', 'EW': 'tab:orange', 'SM': 'tab:green', 'WV': '0.5'}
    f, axs = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    for ax, (side, box) in zip(axs, [('west_of_antimeridian', box_w), ('east_of_antimeridian', box_e)]):
        clipped = plan_gdf[plan_gdf.intersects(box)]
        for mode, sub in clipped.groupby('Mode'):
            sub.plot(ax=ax, color=mode_colors.get(mode, 'k'), label=mode, linewidth=1)
        ax.set_xlim(box.bounds[0], box.bounds[2])
        ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.set_title(f'current ESA acquisition plan -- {side}\n({len(clipped)} planned datatakes)')
        ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No current ESA acquisition-plan KML available for this AOI/time window.')

## Section 10 -- Tile(8,0) vs Tile(8,243): the antimeridian S1-read bug -- symptom, root cause, the one-line upstream fix, and what remains

The Section 3 coarsened plot shows a hard cutoff right after the antimeridian, and it lines up with **Tile(row=8, column=0)** (bounds -179.999 to -178.525) versus its mirror on the other side, **Tile(row=8, column=243)** (bounds 178.319 to 179.794). Both tiles are folded into `test_tiles_gdf` above, so they show up in every combined-view figure in Sections 4b, 5b, and 8.

This section runs the full arc:

1. **Symptom and scope** -- tile (8,0) comes back empty even though the data demonstrably exists, and so does the entire westmost column of the global tile grid.
2. **Root cause** -- [odc-geo #208](https://github.com/opendatacube/odc-geo/issues/208): items in UTM zone 1 get an invalid, world-spanning footprint during item->tile binning and are silently dropped.
3. **The fix** -- a one-line change to odc-stac, submitted upstream as a PR, demonstrated live below.
4. **What remains** -- the sibling antimeridian bugs the PR does *not* fix, the explorations behind `.agents/odc-geo-antimeridian-fix.patch`, and the path forward.


In [ ]:
tile_pair = {'Tile(8,0)': config.get_tile(8, 0), 'Tile(8,243)': config.get_tile(8, 243)}
pix_cols = [c for c in chuk_tiles_gdf.columns if c.startswith('pix_ct_')]

for name, tile in tile_pair.items():
    row = chuk_tiles_gdf[(chuk_tiles_gdf['row'] == tile.row) & (chuk_tiles_gdf['col'] == tile.col)].iloc[0]
    print(f'{name}: bounds {tuple(tile.bbox_gdf.total_bounds.round(4))}, '
          f'percent_valid_snow_pixels={row.get("percent_valid_snow_pixels")}, success={row.get("success")}')
    print('  per-water-year valid S1 pixel count:', {c.replace('pix_ct_', ''): row[c] for c in pix_cols})

In [ ]:
%%time
# Rule out "can't find scenes" first: reproduce the exact search get_sentinel1_rtc does
# (intersects=geobox.geographic_extent) for both tiles over one water year.
tile_pair_items = {}
for name, tile in tile_pair.items():
    gj = tile.geobox.geographic_extent.geojson()
    search = catalog.search(collections=['sentinel-1-rtc'], intersects=gj,
                            datetime=f'{WY_CHECK - 1}-10-01/{WY_CHECK}-09-30', limit=1000)
    items = list(search.item_collection())
    tile_pair_items[name] = items
    modes = pd.Series([it.properties['sar:instrument_mode'] for it in items]).value_counts().to_dict()
    epsgs = pd.Series([it.properties.get('proj:code') for it in items]).value_counts().to_dict()
    print(f'{name}: {len(items):3d} matching sentinel-1-rtc items for WY{WY_CHECK}  modes={modes}  utm={epsgs}')

In [ ]:
%%time
# Item discovery is fine on both sides -- so the problem must be at *read* time. For one item that
# genuinely intersects Tile(8,0), compare three ways of putting its pixels onto the tile's grid:
# (1) the raw asset in its native UTM CRS (is the data actually there?), (2) a plain
# rasterio.warp.reproject "by hand" into the tile geobox, and (3) odc.stac.load into the SAME
# geobox -- what get_sentinel1_rtc (and therefore the whole pipeline) actually calls.
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.windows import from_bounds
from rasterio.enums import Resampling as ReadResampling
import odc.stac

t0 = tile_pair['Tile(8,0)']
small_gb0 = t0.geobox.zoom_out(8)  # coarser grid purely so this cell runs in seconds, not minutes
probe_item = tile_pair_items['Tile(8,0)'][0]
print('probe item:', probe_item.id, probe_item.properties.get('proj:code'))

minx, miny, maxx, maxy = t0.bbox_gdf.total_bounds
with rasterio.open(probe_item.assets['vv'].href) as src:
    b = transform_bounds('EPSG:4326', src.crs, minx, miny, maxx, maxy)
    left, right = max(b[0], src.bounds.left), min(b[2], src.bounds.right)
    bottom, top = max(b[1], src.bounds.bottom), min(b[3], src.bounds.top)
    win = from_bounds(left, bottom, right, top, transform=src.transform)
    out_h, out_w = 500, 500
    src_data = src.read(1, window=win, out_shape=(out_h, out_w), resampling=ReadResampling.average)
    raw_valid_frac = float(np.mean(src_data != src.nodata))
    base_transform = src.window_transform(win)
    decim_transform = base_transform * base_transform.scale(win.width / out_w, win.height / out_h)

    manual_dst = np.full((small_gb0.shape.y, small_gb0.shape.x), -32768, dtype='float32')
    reproject(source=src_data, destination=manual_dst, src_transform=decim_transform, src_crs=src.crs,
             dst_transform=small_gb0.affine, dst_crs=str(small_gb0.crs), dst_nodata=-32768,
             src_nodata=-32768, resampling=Resampling.bilinear)
    manual_valid_frac = float(np.mean(np.isfinite(manual_dst) & (manual_dst != -32768)))

odc_ds = odc.stac.load([probe_item], bands=['vv'], nodata=-32768, geobox=small_gb0,
                       resampling='bilinear', chunks={})
odc_vv = odc_ds['vv'].isel(time=0).values
odc_valid_frac = float(np.mean(np.isfinite(odc_vv) & (odc_vv != -32768)))

print(f'(1) raw asset, native UTM, windowed read        -> valid fraction = {raw_valid_frac:.3f}')
print(f'(2) manual rasterio.warp.reproject into geobox  -> valid fraction = {manual_valid_frac:.3f}')
print(f'(3) odc.stac.load into the SAME geobox           -> valid fraction = {odc_valid_frac:.3f}  <- what the pipeline calls')

f, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].imshow(manual_dst != -32768, cmap='gray', vmin=0, vmax=1)
axs[0].set_title('(2) manual rasterio reproject -- has data')
axs[1].imshow(np.isfinite(odc_vv) & (odc_vv != -32768), cmap='gray', vmin=0, vmax=1)
axs[1].set_title('(3) odc.stac.load (pipeline path) -- empty')
plt.tight_layout()
plt.show()

In [ ]:
%%time
# Mirror check: does odc.stac.load also fail on Tile(8,243) (UTM zone 60N, approaching +180)?
# If not, the bug is specific to the negative-longitude side near -180 (UTM zone 1N), not to
# "any tile that touches the antimeridian".
t243 = tile_pair['Tile(8,243)']
small_gb243 = t243.geobox.zoom_out(8)
probe_item_243 = tile_pair_items['Tile(8,243)'][0]
print('probe item:', probe_item_243.id, probe_item_243.properties.get('proj:code'))

odc_ds_243 = odc.stac.load([probe_item_243], bands=['vv'], nodata=-32768, geobox=small_gb243,
                           resampling='bilinear', chunks={})
odc_vv_243 = odc_ds_243['vv'].isel(time=0).values
print(f'odc.stac.load valid fraction for Tile(8,243), single item -> '
      f'{float(np.mean(np.isfinite(odc_vv_243) & (odc_vv_243 != -32768))):.3f}')

In [ ]:
# Is this a Tile(8,0)-only quirk, or does it hit the whole western edge of the global tile grid?
# No network needed -- config.valid_tiles_gdf already has per-water-year pixel counts for every
# processed tile worldwide.
pix_cols_global = [c for c in config.valid_tiles_gdf.columns if c.startswith('pix_ct_')]
vt = config.valid_tiles_gdf.copy()
vt['total_pix_ct'] = vt[pix_cols_global].sum(axis=1, skipna=True)
vt['any_valid_pixels'] = vt['total_pix_ct'] > 0

print('col  n_tiles  n_with_any_valid_S1_pixels  mean_total_pix_ct  success=True count')
for col in [0, 1, 2, 242, 243, 244]:
    sub = vt[vt['col'] == col]
    if not len(sub):
        print(f'{col:3d}  no valid (seasonal-snow) tiles at this column')
        continue
    print(f'{col:3d}  {len(sub):7d}  {int(sub["any_valid_pixels"].sum()):26d}  '
          f'{sub["total_pix_ct"].mean():17,.0f}  {(sub["success"] == True).sum():d}')

**H5 -- a real S1-read bug, not a data-availability or mask-reprojection issue.** The three-way comparison above shows the raw Sentinel-1 RTC asset genuinely covers Tile(8,0) (>99% valid in its native UTM zone 1N projection), and a plain `rasterio.warp.reproject` correctly carries that data onto the tile's lat/lon grid -- but `odc.stac.load` (what `get_sentinel1_rtc`, and therefore the whole pipeline, actually calls) returns **100% nodata** for the identical item into the identical target grid. The failure holds regardless of resampling method (nearest/bilinear/cubic) and regardless of whether the target grid is passed as `geobox=` or as `bbox=`/`crs=`/`resolution=`; the item's `proj:bbox`/`proj:transform`/`proj:shape` metadata all match the real file exactly, so this isn't stale/wrong STAC metadata either. The failure is specific to target grids sitting at the extreme negative-longitude edge approaching -180 (UTM zone 1N) -- the mirror check on Tile(8,243) (UTM zone 60N, approaching +180) loads normally.

**And it isn't just Tile(8,0):** every one of the 6 processed tiles at `col=0` (the entire westmost column of the global tile grid, at every latitude with seasonal snow) has **zero** valid S1 pixels in **every** water year, despite all 6 being flagged `success=True` -- because `get_sentinel1_rtc` doesn't raise an exception here, it just silently returns an all-nodata array. That's exactly why Section 1's `success`/`error_messages` check never caught this: there's no error to report. `col=1`/`col=2` (still inside UTM zone 1N, one/two tiles further from -180) are partially affected; `col=242-244` (approaching +180 from the west, UTM zone 60N) show only mild degradation by comparison -- so this is asymmetric and concentrated right at the -180 edge, not a general antimeridian effect.

This is a real bug in `odc.stac.load`'s handling of destination grids at the -180 boundary -- and it turns out to be a known (but unfixed) upstream issue. The precise mechanism is traced below: the failure happens one layer *above* the pixel-reading code, during item->tile binning, and the fix is a one-line upstream change to odc-stac (now submitted as a PR, with a live before/after demo further down). No rasterio-based fallback read path is needed.


### Is this a known upstream bug? -- yes, but not the one it first looks like

Checked live against the GitHub issue trackers, two open odc-stac issues look like matches at first glance:

- [**#165 "Data gap when loading items across the antimeridian"**](https://github.com/opendatacube/odc-stac/issues/165) (opened 2024-08-15, still open) -- Copernicus DEM tiles loaded into a geobox crossing the antimeridian come back with a gap.
- [**#172 "Inability to load pixels near/at antimeridian"**](https://github.com/opendatacube/odc-stac/issues/172) (opened 2024-09-05, still open) -- a 256x256 geobox placed immediately adjacent to -180 fails to load Sentinel-2 pixels. Maintainer **alexgleith** traced it to [`odc/geo/overlap.py`](https://github.com/opendatacube/odc-geo/blob/94ad126571cdddcf4884b0f9ea4024dde15d0208/odc/geo/overlap.py#L138) -- the `GbxPointTransform`/`native_pix_transform` code that computes the per-pixel *read window*, where transformed boundary points land on the wrong side of the +/-180 branch cut. His last comment: *"At the core, the issue is caused by code in odc-geo but I can't think of any way to fix the issue!"*

**But our case is not the #172 read-window bug.** Calling the exact functions that issue implicates (`native_pix_transform`, `compute_reproject_roi`) directly against our real Tile(8,0) source/destination geoboxes returns *sane, non-degenerate* ROIs -- and instrumenting a scratch copy of `odc.loader._rio` shows our failing `odc.stac.load(..., chunks={})` never even reaches the pixel-reading code. Whatever discards the data for us happens earlier, during item->tile binning. That earlier failure is traced precisely in the next cell (it matches [odc-geo #208](https://github.com/opendatacube/odc-geo/issues/208)); #165 and #172 are picked back up at the end of this section -- they are real sibling bugs in the same family, just not ours.


### Precise root cause, fully traced -- and it IS the same bug as an existing, still-open issue

Digging further (using `odc.stac.load(..., debug=True)`, which exposes the internal `tyx_bins` -- which item got binned into which destination tile -- directly): for Tile(8,0), `tyx_bins` comes back **completely empty**. The item never gets assigned to *any* destination tile, so no read is ever attempted -- that's why nothing in `_rio.py` (the actual pixel-reading code) ever gets called, and why the earlier three-way raw/manual/odc comparison couldn't reproduce the failure by calling those functions directly: the real failure happens one layer *before* any of that.

**The exact mechanism:**

1. To decide which destination tile(s) an item overlaps, `odc.stac` calls   `ParsedItem.safe_geometry(dst_crs)`, which prefers the item's *full-image footprint from proj   metadata* over the plain STAC-declared geometry (`odc/stac/model.py`, `image_geometry`).
2. That footprint comes from `GeoBox.footprint(dst_crs)`, called on the item's **native UTM geobox**   (EPSG:32601 for our Tile(8,0) source) -- i.e. it reprojects the UTM rectangle's boundary into   EPSG:4326.
3. For a UTM zone abutting +/-180 (zone 1N here), this reprojection is **not** antimeridian-aware by   default and produces an **invalid, "world-spanning" polygon** -- verified directly: bounds   `-179.97` to `+179.97`, `is_valid=False`, with boundary vertices landing on the *wrong* side of the   dateline (e.g. a vertex near `+177` that should be near `-179`).
4. That broken polygon is then tested against the destination `GeoboxTiles` grid via a `shapely`/GEOS   `disjoint()` check. For an invalid, self-intersecting, world-spanning polygon, this check comes back   **wrongly `True`** for the one destination tile that genuinely does overlap -- so the item is silently   dropped from `tyx_bins`, with no exception, no warning, nothing. Every S1 scene over Tile(8,0) hits   this the same way, which is why the tile is empty in *every* water year.

**This turns out to already be reported, with the root cause independently pinpointed by odc-geo's own lead maintainer:** [**odc-geo #208 -- "Inability to load some data near antimeridian into WGS84"**](https://github.com/opendatacube/odc-geo/issues/208) (opened 2025-02-26, **still open**, most recent comment 2025-12-12). Maintainer **Kirill888**: *"brokenness starts in `GeoBox.footprint` for the geobox of that data that spans `lon=180` ... this is relevant code path in `parse_item(item).safe_geometry(4326)`"* -- exactly the call chain traced above, independently arrived at from the odc-geo side. So this is **not unique to our pipeline or to Sentinel-1** -- it hits any STAC-driven `odc.stac.load()` call where a source asset's UTM zone abuts the antimeridian and the destination grid is geographic (WGS84).

**A fix already exists in odc-geo, but isn't wired up in odc-stac.** A related odc-geo issue, [**#234 -- "Fix GEOSException when reprojecting antimeridian-spanning geometries"**](https://github.com/opendatacube/odc-geo/issues/234) (closed 2025-07-07), added a `wrapdateline` parameter to `GeoBox.footprint()` specifically for this. Tested directly against our real source geobox: `footprint('EPSG:4326', wrapdateline=True)` returns a **valid** `MultiPolygon` (correctly split at the dateline), and re-running `GeoboxTiles.tiles()` with it correctly finds Tile(8,0). The catch: `odc.stac`'s `ParsedItem.image_geometry()` (installed version: odc-stac 0.5.2) still calls `gbox.footprint(crs)` **without** `wrapdateline=True` -- the odc-geo fix from #234 was never threaded through on the consuming (odc-stac) side, so #208 remains unresolved in practice. odc-geo's own maintainer view (Kirill888, same thread) is that a truly general fix needs a deeper rework -- comparing GeoBoxes directly (disjoint / overlapping / not-sure) rather than via lossy polygon-transform intermediates -- since blanket dateline-wrapping "can not be made safe enough in a general case." For our specific case (standard UTM source, geographic destination) it tested completely safe, see below.

**End-to-end confirmation:** monkey-patching just `ParsedItem.image_geometry` to add `wrapdateline=True` to its one `gbox.footprint(...)` call, then re-running the *exact* failing `odc.stac.load(...)` call for Tile(8,0), fixes it completely: `tyx_bins` goes from `{}` to `{(0, 0, 0): [0]}`, and the loaded data goes from 0% to **100%** valid. And for a normal, non-antimeridian UTM geobox (tested against a control zone-33N case), `wrapdateline=True` produces an **identical** result to the default -- so this specific one-line change carries no evident regression risk for the ordinary case.

**So, to directly answer "is ours unique?": no.** It's the same bug as odc-geo #208, we've now traced the exact mechanism odc-geo's own maintainer only partially identified, confirmed odc-geo already shipped (but odc-stac never adopted) the fix, and validated that fix resolves our exact failure end-to-end. What's genuinely new here relative to the existing issue threads is the full mechanistic trace (`tyx_bins` -> `safe_geometry` -> `image_geometry` -> `GeoBox.footprint` -> invalid polygon -> wrong `disjoint()` result) plus a concrete Sentinel-1/RTC reproducer and a verified one-line patch -- which is more complete than what's currently on either open issue.

### The actual fix: one line in odc-stac (upstream PR)

Given the trace above, the fix needs nothing heavier than what odc-geo already ships: **odc-geo has handled the UTM-abutting-the-antimeridian footprint correctly since 0.5.0**, via the `wrapdateline=True` option added to `GeoBox.footprint()` in [opendatacube/odc-geo#234](https://github.com/opendatacube/odc-geo/pull/234) -- odc-stac just never asks for it. The complete fix is one line in `odc/stac/model.py`'s `ParsedItem.image_geometry()`:

```python
return gbox.footprint(crs)                      # before
return gbox.footprint(crs, wrapdateline=True)   # after
```

plus a pin bump (`odc-geo>=0.4.7` -> `>=0.5.0`) and a regression test. That is exactly what the upstream odc-stac PR submits (branch `footprint-wrapdateline`; full text and verification record in `.agents/odc-stac-pr-draft.md` and `.agents/minimal-fix-note-2026-07-20.md`). `wrapdateline` is a no-op unless the destination CRS is geographic, so the change cannot affect projected-destination loads.

The cells below are the live before/after demonstration -- the same minimal example that went into the PR. The odc-stac installed in this environment is still stock, so the first load fails exactly the way the pipeline did; the second applies the PR's change as a runtime monkeypatch (byte-equivalent to the patched method) and the identical load succeeds.

> **Run order matters:** run these demo cells *before* the exploration cells further down -- those monkeypatch other parts of the odc stack (`GbxPointTransform`, `CRS.transformer_to_crs`) and would contaminate the "stock" BEFORE picture.


In [ ]:
%%time
# Minimal reproducer (same as in the upstream PR): one S1 RTC item over Chukotka (UTM zone 1N),
# loaded into a small EPSG:4326 geobox right at -180 -- Tile(8,0)'s northwest corner. Stock
# odc-stac, so this is the BEFORE picture. debug=True exposes tyx_bins, the item->tile binning.
from affine import Affine
from odc.geo.geobox import GeoBox

s1_item = next(catalog.search(
    collections=['sentinel-1-rtc'],
    ids=['S1B_IW_GRDH_1SDV_20200921T183011_20200921T183040_023477_02C993_rtc'],
).items())

demo_gb = GeoBox((256, 256),
                 Affine(0.00576000576, 0.0, -179.99945999946, 0.0, -0.00576000576, 69.3029493),
                 'EPSG:4326')

ds_before = odc.stac.load([s1_item], bands=['vv'], nodata=-32768, geobox=demo_gb,
                          resampling='bilinear', chunks={}, debug=True)
vv_before = ds_before['vv'].isel(time=0).values
print('tyx_bins       :', dict(ds_before.encoding['debug'].tyx_bins))
print('valid fraction :', float(np.mean(np.isfinite(vv_before) & (vv_before != -32768))))


In [ ]:
%%time
# The PR's change, applied as a runtime monkeypatch (byte-equivalent to the patched
# ParsedItem.image_geometry). This is also the pipeline workaround until an odc-stac
# release ships the fix.
import odc.stac.model as _osm

def _image_geometry_wrapdateline(self, crs=_osm.Unset(), bands=None):
    if isinstance(crs, _osm.Unset):
        crs = None
    for gbox in self.geoboxes(bands):
        if gbox.crs is not None:
            if crs is None or crs == gbox.crs:
                return gbox.extent
            return gbox.footprint(crs, wrapdateline=True)  # <-- the one-line fix
    return None

_osm.ParsedItem.image_geometry = _image_geometry_wrapdateline

ds_after = odc.stac.load([s1_item], bands=['vv'], nodata=-32768, geobox=demo_gb,
                         resampling='bilinear', chunks={}, debug=True)
vv_after = ds_after['vv'].isel(time=0).values
print('tyx_bins       :', dict(ds_after.encoding['debug'].tyx_bins))
print('valid fraction :', float(np.mean(np.isfinite(vv_after) & (vv_after != -32768))))


In [ ]:
# Side-by-side: same item, same geobox, same installed odc-geo -- only the one-line change differs.
def _to_db(a):
    a = np.where((a == -32768) | ~np.isfinite(a) | (a <= 0), np.nan, a)
    return 10 * np.log10(a)

bb = demo_gb.boundingbox
f, axs = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax, vv, title in [
        (axs[0], vv_before, 'WITHOUT wrapdateline=True\nin image_geometry() (stock odc-stac)'),
        (axs[1], vv_after, 'WITH wrapdateline=True\nin image_geometry() (the PR / monkeypatch)')]:
    db = _to_db(vv)
    ax.set_facecolor('0.85')
    im = ax.imshow(db, extent=[bb.left, bb.right, bb.bottom, bb.top], cmap='gray', vmin=-25, vmax=5)
    valid = np.mean(np.isfinite(db))
    if valid == 0:
        ax.text(0.5, 0.5, 'all nodata\n(item silently dropped during binning)',
                transform=ax.transAxes, ha='center', va='center', color='0.35', fontsize=12)
    ax.text(0.98, 0.02, f'{valid:.0%} valid pixels', transform=ax.transAxes, ha='right', va='bottom',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
    ax.set_title(title)
    ax.set_xlabel('longitude')
axs[0].set_ylabel('latitude')
f.colorbar(im, ax=axs, shrink=0.8, label='VV backscatter (dB)');


### Two gotchas that made this bug hard to see (and easy to mis-verify)

1. **The nodata fill lies to `isnull()`.** With `nodata=-32768` (or the value picked up from the
   item's raster metadata), a failed load comes back *filled with -32768, not NaN* -- so
   `ds.vv.isnull().mean()` reports 0% missing on an array that is 100% nodata. Any check here has
   to mask the fill value explicitly (as the cells above do), or the bug verifies as "fixed" when
   it isn't.
2. **The failure is geobox-sensitive.** Whether the invalid world-spanning footprint "accidentally"
   passes the `disjoint()` test depends on the exact destination geobox: shifting this same window
   to start at exactly -180 (`GeoBox.from_bbox((-180, 67.8, -178.5, 69.3), ...)`) happens to load
   fine on stock odc-stac. Nudging the window by a fraction of a pixel can make the problem come
   and go -- which is why it read as a "data gap" for so long, and why casual spot checks near the
   antimeridian can miss it entirely.

### Verification record (2026-07-20/21)

- End-to-end S1 reproducer above: valid fraction **0.0 -> 1.0**, `tyx_bins` `{}` -> `{(0,0,0): [0]}`.
- odc-geo #208's own Landsat reproducer (earth-search `LC08_L2SR_074071_20241228_02_T1`,
  WGS84GRID30): footprint invalid (757 deg^2, 0 tiles matched) -> valid MultiPolygon (4.6 deg^2,
  9 tiles matched).
- Patched odc-stac test suite: identical results to main plus the new regression test
  (`test_image_geometry_antimeridian` -- fails on main, passes with the fix). One pre-existing
  unrelated failure both ways (`test_stac_load_smoketest`, a local datetime64 env-skew issue).
- The monkeypatch route itself was validated against the real Tile(8,0) pipeline case earlier in this section.


### Is `wrapdateline=True` a *comprehensive* fix? No -- it has the same limitation odc-geo's own docstring admits to

*(Everything from here down explores the wider upstream antimeridian bug family. None of it is needed for the pipeline fix above -- the one-liner is complete for our failure mode -- but it maps what the PR does and doesn't cover, and it's where the `.agents/odc-geo-antimeridian-fix.patch` prototype came from.)*

`GeoBox.footprint()`'s docstring for `wrapdateline` is refreshingly honest: *"Attempt to gracefully handle geometry that intersects the dateline when converting to geographic projections. **Currently only works in few specific cases (source CRS is smooth over the dateline).**"* Internally it calls `chop_along_antimeridian()`, which projects the lon=180 meridian *into the source CRS* and splits the source geometry along that line *before* reprojecting. That works well for UTM/sinusoidal (a genuinely "smooth" source CRS crossing a well-defined line) -- but it has no concept of a **pole**.

A source geobox in a polar-aspect projection (e.g. NSIDC Sea Ice Polar Stereographic, EPSG:3413/3031) that genuinely *contains the pole itself* isn't a "crosses one meridian" problem at all -- every meridian, including the antimeridian, converges at that single point. Chopping along "the lon=180 line" doesn't resolve that. Tested directly:

In [ ]:
from odc.geo.geobox import GeoBox, GeoboxTiles
from affine import Affine

# A polar-stereographic tile that genuinely includes the North Pole (NSIDC Sea Ice Polar
# Stereographic North, EPSG:3413) -- a legitimate, non-buggy case: every longitude really does
# converge inside this tile.
polar_gb = GeoBox((2000, 2000), Affine(1000.0, 0.0, -1000000.0, 0.0, -1000.0, 1000000.0), 'EPSG:3413')

fp_default = polar_gb.footprint('EPSG:4326')
fp_wrap = polar_gb.footprint('EPSG:4326', wrapdateline=True)
print('default          : valid =', fp_default.geom.is_valid, ' bounds =', fp_default.boundingbox)
print('wrapdateline=True: valid =', fp_wrap.geom.is_valid, ' bounds =', fp_wrap.boundingbox)

# Does this invalid geometry actually break tile matching, the way it did for our Tile(8,0) case?
dst_near_pole = GeoBox((256, 256), Affine(0.01, 0.0, -179.995, 0.0, -0.01, 90.0), 'EPSG:4326')
dst_far = GeoBox((256, 256), Affine(0.01, 0.0, 10.0, 0.0, -0.01, 50.0), 'EPSG:4326')
gbt_near = GeoboxTiles(dst_near_pole, (256, 256))
gbt_far = GeoboxTiles(dst_far, (256, 256))
print('near-pole dest tile matched (should be [(0, 0)]):', list(gbt_near.tiles(fp_wrap)))
print('far dest tile matched (should be []):            ', list(gbt_far.tiles(fp_wrap)))

`wrapdateline=True` leaves the polar case just as invalid as the default, and the practical consequence is the same silent failure as Tile(8,0): the near-pole destination tile -- which genuinely should match -- comes back **empty**, a false negative. This matters beyond a synthetic test case: polar-stereographic tiling is exactly how real Arctic/Antarctic products are commonly distributed (sea ice concentration, SAR EW-mode composites, passive microwave), so this isn't an edge case invented for this notebook.

### A comprehensive fix: the `antimeridian` package handles both failure modes

odc-stac maintainer alexgleith speculated in the #208 thread that this "could be resolved by using the [antimeridian](https://github.com/gadomski/antimeridian) fix" -- a small, purpose-built, actively-maintained package (not an odc dependency currently) for exactly this class of problem. That suggestion doesn't appear to have been tested against the pole-enclosing case before. It was tested here (requires `pip install antimeridian`; not currently part of this project's environment):

In [ ]:
# Requires: pip install antimeridian  (not currently a dependency of this project)
import antimeridian
from odc.geo.geom import Geometry

test_cases = {
    'UTM zone1N (Tile(8,0) source, dateline)': GeoBox(
        (25129, 29486), Affine(10.0, 0.0, 281870.0, 0.0, -10.0, 7738430.0), 'EPSG:32601'),
    'polar stereo north (pole-enclosing)': polar_gb,
    'polar stereo south (pole-enclosing)': GeoBox(
        (2000, 2000), Affine(1000.0, 0.0, -1000000.0, 0.0, -1000.0, 1000000.0), 'EPSG:3031'),
}

for name, gb in test_cases.items():
    naive = gb.footprint('EPSG:4326')  # plain/default transform -- may be invalid
    fixed_shape = antimeridian.fix_shape(naive.geom.__geo_interface__)
    fixed = Geometry(fixed_shape, crs='EPSG:4326')
    print(f'{name}:')
    print(f'   naive : valid={naive.geom.is_valid}  bounds={naive.boundingbox}')
    print(f'   fixed : valid={fixed.geom.is_valid}  type={fixed.geom.geom_type}  bounds={fixed.boundingbox}')

`antimeridian.fix_polygon()`/`fix_shape()` correctly repairs **every** case above with no special flags needed (it auto-detects both ordinary dateline crossings and pole enclosure) -- something odc-geo's own `wrapdateline=True` cannot do. It also doesn't need the boundary pre-densified (odc-geo densifies via the `resolution=` parameter before chopping; `antimeridian` works directly off as few as 4 corner points, since it reasons about meridian crossings per-edge rather than by sampling density).

### End-to-end proof against the real failing case

Patching just `ParsedItem.image_geometry()` to run the naive footprint through `antimeridian.fix_shape()` (instead of odc-geo's own wrapdateline logic) and re-running the *exact* failing `odc.stac.load()` call from earlier fixes it completely -- tested in an isolated venv built on top of this project's actual environment (`pixi`'s `.pixi/envs/default` plus `antimeridian` layered on top via `--system-site-packages`, so the real `odc.stac`/`odc.geo`/`pystac_client` versions installed here were exercised, not a fresh unrelated install):

```python
import antimeridian
import odc.stac.model as model
from odc.geo.geom import Geometry

def patched_image_geometry(self, crs=None, bands=None):
    for gbox in self.geoboxes(bands):
        if gbox.crs is not None:
            if crs is None or crs == gbox.crs:
                return gbox.extent
            naive = gbox.footprint(crs)
            fixed = antimeridian.fix_shape(naive.geom.__geo_interface__)
            return Geometry(fixed, crs=crs)
    return None
model.ParsedItem.image_geometry = patched_image_geometry

# re-run the Tile(8,0) odc.stac.load(...) reproducer from earlier:
# BEFORE: tyx_bins = {}                      valid fraction = 0.0
# AFTER:  tyx_bins = {(0, 0, 0): [0]}         valid fraction = 1.0
```

And the polar case, run through the same fix and fed into `GeoboxTiles.tiles()` the way `odc.stac` actually uses it:

```python
# BEFORE (wrapdateline=True): near-pole dest tile matched = []      (false negative)
# AFTER  (antimeridian fix):  near-pole dest tile matched = [(0, 0)] (correct)
#                             far dest tile matched        = []      (still correctly empty -- no false positives)
```

### Toward a comprehensive, low-risk fix

Two remaining practical concerns, both addressable:

1. **Performance.** `antimeridian.fix_polygon()` costs about the same per-vertex as odc-geo's current   approach (~8-9ms for a ~400-point densified boundary; ~0.1ms for a handful of corner points) -- it   is *not* magically free for large/complex geometries. But it **doesn't need to run on every item**:   the naive/default `to_crs()` transform is cheap, and whether it produced something broken   (`not naive.geom.is_valid`, or a bounding box spanning close to the full +/-180 or +/-90 range) is a   cheap check to make *after* the fact. Only items whose naive transform actually comes out broken   need the more expensive repair -- the overwhelming majority of items (nowhere near a dateline or   pole) pay nothing extra.
2. **Dependency.** `antimeridian` is a small, focused, MIT-licensed package with few transitive   dependencies (shapely, numpy) -- a reasonable optional dependency, or could be vendored/reimplemented   if odc-geo prefers not to add it.

Putting this together, a fix along these lines in `odc/geo/geobox.py`'s `GeoBox.footprint()` (or `Geometry.to_crs()`) would plausibly need **no opt-in flag at all** -- it could safely become the unconditional default:

```python
def footprint(self, crs, ...):
    naive = self.extent.to_crs(crs, resolution=self._reproject_resolution(npoints))
    if not naive.geom.is_valid:  # only the rare antimeridian/pole-adjacent case pays this cost
        try:
            import antimeridian
            return Geometry(antimeridian.fix_shape(naive.geom.__geo_interface__), crs=naive.crs)
        except ImportError:
            pass  # fall back to today's behavior if antimeridian isn't installed
    return naive
```

Wiring this into `ParsedItem.image_geometry()`/`safe_geometry()` (`odc/stac/model.py`) would close #208 for the dateline case *and* the pole case it never covered, with no measurable cost for the common case and no new required dependency for users who don't have `antimeridian` installed. This is deliberately **not** what the odc-stac PR above does -- that PR stays minimal and mergeable, since `wrapdateline` already covers the dateline case that actually bit this pipeline -- but it is the natural direction if upstream ever wants full coverage, pole case included.

### Is this the same as odc-geo #176 / #234 / #23?

Checked all three directly:

- **[#176](https://github.com/opendatacube/odc-geo/issues/176)** ("Some cases of geometries that are near and that cross the antimeridian don't work well in `odc.geo.overlap._relative_rois`") -- a **different, sibling bug**, not fixed by anything above. It lives in `odc/geo/overlap.py`'s `GbxPointTransform`/`native_pix_transform` -- the code used to compute the *per-pixel read window* during an actual reprojection (`compute_reproject_roi`, called from `odc.loader._rio`), as opposed to `GeoBox.footprint()`, which is used for *item-to-tile spatial binning* (our bug, #208). Confirmed by re-running #176's exact original reproducer against the currently-installed odc-geo: it still reproduces the exact reported symptom (a transformed coordinate of `262143.98348895763` where it should be near 0) -- unchanged since the issue was filed.
- **[#234](https://github.com/opendatacube/odc-geo/pull/234)** -- confirmed via its file diff (`odc/geo/geobox.py` +53/-5, `tests/test_antimeridian.py` +65) that it never touches `odc/geo/overlap.py` at all, so it was never intended to (and doesn't) fix #176. It's the same PR already discussed above (added `wrapdateline` to `footprint()` and error-handling to `GeoboxTiles.grid_intersect()`) -- relevant to #208, not to #176.
- **[#23](https://github.com/opendatacube/odc-geo/pull/23)** ("Preparing for rc0") -- from 2022, general early-stage feature work (GeoBox-from-bbox construction, visualization/svg output, misc convenience additions). Doesn't appear related to the antimeridian family of issues at all.

But: `#176`'s own thread mixes **two different reproducers**. Its second example (a Landsat item loaded into a destination geobox with its origin placed at exactly `lon=180`) was tested directly and turns out to hit the *same* `tyx_bins`-empty binning failure as our Tile(8,0) case -- not the `_relative_rois` point-transform bug its first example demonstrates. So #176 isn't one bug, it's two, filed together.

In [ ]:
# Re-run odc-geo #176's ORIGINAL minimal reproducer (point-transform ROI computation, not the
# footprint/binning path) against the currently-installed odc-geo, to check whether it's still broken.
import numpy as np
from odc.geo.types import xy_
from odc.geo.overlap import (
    roi_boundary, unstack_xy, stack_xy, gbox_boundary, roi_from_points, native_pix_transform)

pts_per_side, padding, align = 5, 1, True
dst_176 = GeoBox((256, 256), Affine(152.87405657034833, 0.0, -20037508.342789244,
                                    0.0, -152.87405657034833, -1995923.6825825237), 'EPSG:3857')
src_176 = GeoBox((10980, 10980), Affine(10.0, 0.0, 99960.0, 0.0, -10.0, 8100040.0), 'EPSG:32701')

tr = native_pix_transform(src_176, dst_176)
xy = tr.back(unstack_xy(gbox_boundary(dst_176, pts_per_side)))
roi_src = roi_from_points(stack_xy(xy), src_176.shape, padding, align=align)
xy_pix_src = unstack_xy(roi_boundary(roi_src, pts_per_side))
xx, yy = np.asarray([pt.xy for pt in xy_pix_src]).T
xys = tr([xy_(x, y) for x, y in zip(xx, yy)])
print('still broken today -- first transformed coordinate (should be near 0):', xys[0].xy)

Confirmed: unchanged since 2024. Two actual fix attempts exist for #176, and both were **closed without being merged**:
[**PR #177 "Set force over flag"**](https://github.com/opendatacube/odc-geo/pull/177) (work in progress, closed 2024-09-26) and [**PR #183 "Add antimeridian test for roi extraction. Currently failing."**](https://github.com/opendatacube/odc-geo/pull/183) (test-only, explicitly "currently failing," closed unmerged 2026-02-16). Neither PR has any substantive review comments -- just bot noise (Netlify/Codecov) -- so they read as abandoned for lack of follow-through, not because the approach was shown not to work.

#177's idea: pass pyproj's `force_over=True` to the `Transformer` used inside `GbxPointTransform` (via a new `force_over` parameter threaded through `CRS.transformer_to_crs`). `force_over` tells PROJ not to wrap/clip a periodic output coordinate back into its canonical range -- exactly the ambiguity causing points to land on the wrong side of the branch cut. Tested it directly against #176's exact reproducer:

In [ ]:
# Test PR #177's approach directly: does pyproj's force_over=True fix #176's reproducer?
# NOTE: this monkeypatch persists for the rest of the kernel session -- run the
# stock before/after demo cells above FIRST.
from pyproj import Transformer as PyprojTransformer
import odc.geo.overlap as overlap_mod

def patched_gbx_init(self, src, dst, back=None):
    assert src.crs is not None and dst.crs is not None
    self._src, self._dst, self._back = src, dst, back
    pt = PyprojTransformer.from_crs(src.crs._crs, dst.crs._crs, always_xy=True, force_over=True)
    self._tr = lambda xx, yy: pt.transform(xx, yy)
    self._clamps = ((-180, 180), (-90, 90)) if src.crs.geographic else None
overlap_mod.GbxPointTransform.__init__ = patched_gbx_init

tr_fixed = overlap_mod.native_pix_transform(src_176, dst_176)
xy_fixed = tr_fixed.back(unstack_xy(gbox_boundary(dst_176, pts_per_side)))
roi_src_fixed = roi_from_points(stack_xy(xy_fixed), src_176.shape, padding, align=align)
xy_pix_src_fixed = unstack_xy(roi_boundary(roi_src_fixed, pts_per_side))
xxf, yyf = np.asarray([pt.xy for pt in xy_pix_src_fixed]).T
xys_fixed = tr_fixed([xy_(x, y) for x, y in zip(xxf, yyf)])
print('force_over=True   -- first transformed coordinate (should be near 0):', xys_fixed[0].xy)
print('roi_src (sane, non-degenerate):', roi_src_fixed, ' vs source shape', src_176.shape)

**`force_over=True` fixes #176's reproducer completely** -- the first coordinate goes from the reported `262143.98` to `-0.017` (matching exactly what the original bug report says it "should" be), and every other previously-wrong point is corrected the same way, with all the already-correct points unchanged. And unlike `wrapdateline`/`antimeridian`, this fix is applied at the *coordinate transform* level itself (`pyproj.Transformer.from_crs(..., force_over=True)`), not by post-hoc repairing an already-broken polygon -- so it is a plausible single fix for the *shared root cause* behind both #176 and #208, rather than two separate patches. Checked directly:

- **No regression on ordinary (non-antimeridian) transforms**: a plain UTM-zone-33N-to-WGS84 transform  gives bit-identical output with or without `force_over=True`.
- **No measurable performance cost**: transformer construction ~0.12ms either way; a transform call  ~3.26 vs 3.27 microseconds. Unlike the `antimeridian`-library approach (several ms per call,  scaling with vertex count), this is effectively free.
- **It also fixes our #208 case** -- applying `force_over=True` to the same `CRS.transformer_to_crs`  used by `GeoBox.footprint()` makes Tile(8,0)'s footprint come back as a **valid, ordinary `Polygon`**  (not even needing the `MultiPolygon` split `antimeridian`/`wrapdateline` produce) and correctly  matches via `GeoboxTiles.tiles()`. One check below.

In [ ]:
# Does force_over=True (applied to the shared CRS.transformer_to_crs) also fix our #208
# footprint/tile-binning case -- and does it also cover the polar pole-enclosing case?
# NOTE: this monkeypatch also persists for the rest of the kernel session.
import odc.geo.crs as crs_mod

def patched_transformer_to_crs(self, other, always_xy=True):
    other = crs_mod.CRS(other) if not isinstance(other, crs_mod.CRS) else other
    pt = PyprojTransformer.from_crs(self._crs, other._crs, always_xy=always_xy, force_over=True)
    return lambda x, y, **kw: pt.transform(x, y, **kw)
crs_mod.CRS.transformer_to_crs = patched_transformer_to_crs

utm_gb_176 = GeoBox((25129, 29486), Affine(10.0, 0.0, 281870.0, 0.0, -10.0, 7738430.0), 'EPSG:32601')
fp_utm = utm_gb_176.footprint('EPSG:4326')
print('Tile(8,0) source footprint, force_over=True:',
      'valid=', fp_utm.geom.is_valid, ' type=', fp_utm.geom.geom_type, ' bounds=', fp_utm.boundingbox)
gbt0 = GeoboxTiles(small_gb0, (256, 256))
print('  tiles matched:', list(gbt0.tiles(fp_utm)), '(should be [(0, 0)])')

fp_polar = polar_gb.footprint('EPSG:4326')
print('polar (pole-enclosing) footprint, force_over=True:',
      'valid=', fp_polar.geom.is_valid, ' bounds=', fp_polar.boundingbox)
print('  -- force_over does NOT reach the pole-enclosing case; antimeridian-library fallback still needed there.')

### A possible future odc-geo PR: `force_over=True` plus a pole-aware fallback

Putting the explorations above together: two independent root causes, one cheap unified fix for the common one plus a targeted fallback for the rare one.

1. **Ordinary antimeridian branch-cut ambiguity** (UTM zones abutting +/-180, sinusoidal, Web-Mercator-   adjacent cases) -- causes *both* #176 (per-pixel ROI computation, `GbxPointTransform`) and #208   (item-to-tile binning, `GeoBox.footprint`), since both ultimately call the same   `CRS.transformer_to_crs`. **Fix: pass `force_over=True` through to the underlying   `pyproj.Transformer.from_crs()` call, unconditionally.** Zero regression, no measurable cost,   already prototyped (unmerged) in PR #177 -- it just needs finishing and wiring into   `GeoBox.footprint()`/`image_geometry()` too, which #177 never touched.
2. **Pole-enclosing singularity** (polar-stereographic tiles containing an actual pole) -- a distinct   geometric situation `force_over` cannot fix (verified above: footprint stays invalid).   **Fix: fall back to `antimeridian.fix_shape()`** (or equivalent pole-aware logic), gated behind   `if not footprint.geom.is_valid`, so the -- comparatively expensive and dependency-adding -- library   call is only reached for the rare case that genuinely needs it.

```python
def footprint(self, crs, ...):
    naive = self.extent.to_crs(crs, resolution=self._reproject_resolution(npoints))  # now force_over=True internally
    if not naive.geom.is_valid:  # only the rare pole-enclosing case reaches here
        try:
            import antimeridian
            return Geometry(antimeridian.fix_shape(naive.geom.__geo_interface__), crs=naive.crs)
        except ImportError:
            pass
    return naive
```

And in `GbxPointTransform.__init__` (`odc/geo/overlap.py`), simply always request `force_over=True` when building `self._tr` -- no gating needed there, since it is free and has no downside observed so far.

A full, tested patch along these lines (`force_over` threaded through `CRS.transformer_to_crs`, the `antimeridian` fallback in `footprint()`, and 5 tests) sits in `.agents/odc-geo-antimeridian-fix.patch`. It is deliberately kept **out** of the odc-stac PR above -- that one stays minimal -- but it is ready to become its own odc-geo PR targeting #176 (and plausibly #165) whenever we choose to push it.


### Status of the antimeridian bug family, and the path forward

The one-line PR fixes *this pipeline's* failure mode, but it is one member of a family of antimeridian bugs in the odc stack. Status of each, all verified live against reproducers in this section:

| issue | where it lives | failure mode | fixed by the odc-stac PR? |
| --- | --- | --- | --- |
| [odc-geo #208](https://github.com/opendatacube/odc-geo/issues/208) | `GeoBox.footprint()` -> item/tile binning | invalid world-spanning footprint -> item silently dropped when the destination CRS is geographic | **yes** (this is our bug) |
| [odc-stac #172](https://github.com/opendatacube/odc-stac/issues/172) / [odc-geo #176](https://github.com/opendatacube/odc-geo/issues/176) | `GbxPointTransform` / `native_pix_transform` (per-pixel read window) | destination boundary points land on the wrong side of the +/-180 branch cut -> wrong/empty read window (projected destinations, e.g. EPSG:3857) | no -- different layer; binning there already works "by luck". `force_over=True` fixes its reproducer (demonstrated above) |
| [odc-stac #165](https://github.com/opendatacube/odc-stac/issues/165) | read path, EPSG:3832 (Pacific-centred Mercator) destination | binning verified OK (`tyx_bins` populated on stock) -- the gap appears at the pixel-read stage, i.e. the #176 class, not the #208 class | no -- `wrapdateline` is a no-op for projected destinations (the PR says "Related", not "Fixes") |
| pole-enclosing footprints (e.g. EPSG:3413/3031 tiles containing a pole) | `GeoBox.footprint()` | footprint invalid and `wrapdateline` cannot repair it (demonstrated above) -- same silent-drop consequence; the `antimeridian` package can | no |

**Path forward:**

1. **Upstream (in flight):** the odc-stac PR (`footprint-wrapdateline` branch), plus a short comment
   on odc-geo #208 linking it. Once merged and released, bump this project's odc-stac pin and delete
   the workaround.
2. **Pipeline (now):** apply the monkeypatch above inside `get_sentinel1_rtc()` and reprocess the
   affected tiles -- all 6 processed `col=0` tiles are 100% empty in every water year while flagged
   `success=True`, and `col=1`/`col=2` were partially affected and need a re-check after
   reprocessing.
3. **Later, optional:** finish the `force_over` odc-geo patch
   (`.agents/odc-geo-antimeridian-fix.patch`) as its own PR targeting #176 -- and plausibly #165,
   though that is not yet confirmed: the 4-configuration pixel-read matrix (stock / patched odc-stac
   / patched odc-geo / both) was still grinding through slow Copernicus-DEM S3 reads when this was
   written. The pole-enclosing case only matters if we ever bin polar-stereographic sources; park it
   unless that happens.
4. **Not an antimeridian bug, but documented here:** EW-mode acquisitions remain invisible to the
   IW-only `sentinel-1-rtc` collection (Section 9) -- a genuine data limitation over Chukotka,
   independent of everything above.


## Interpretation guide

- **S1 counts collapse at the same longitudes as the runoff gap** -> H1: underlying Sentinel-1 RTC availability. Nothing to fix in the pipeline; document as a data limitation (or look for another RTC source for that region).
- **Legacy sinusoidal MODIS store already empty over Chukotka (Section 4 native plots)** -> H2: the seasonal snow mask input is the gap; check whether the new `MODIS_snow_phenology` store fixes it (Section 5) and consider re-running affected tiles against the new mask.
- **Sinusoidal store has data, but the Section 4b tile replication comes back empty/garbled/failed** -> H3: pipeline reprojection. Suspects: `rio.clip_box(*bounds, crs='EPSG:4326')` transforming a 4326 box that touches +/-180 into a degenerate/wrong-side sinusoidal window, and `reproject_match` wrapping across the projection edge. Compare the `rasterio` vs `odc` panels — if `odc` is clean where `rasterio` is broken (or vice versa), that's the fix; the control tile (~170E) shows what "healthy" looks like.
- **Full-res has data where the coarsened store doesn't (Section 3 overlay)** -> H4: coarsening/visualization stage; check the coarsening notebook's handling of the lon edges.
- Also check Section 1: failed/unattempted tiles with reprojection-flavored `error_messages` short-circuit everything above.
- **Section 9 -- acquisition mode (IW/EW)**: Planetary Computer's `sentinel-1-rtc` collection is IW-only; `sentinel-1-grd` shows Chukotka gets a real, seasonally-cycling share of EW acquisitions every water year (not a one-time cutover, and not antimeridian-specific -- similar on both sides of 180). Refines H1: part of the record is invisible to the pipeline purely because of acquisition mode, independent of tile/reprojection logic.
- **Section 10 -- S1-read bug at -180 (H5), root-caused and fixed upstream**: `odc.stac.load` (used by `get_sentinel1_rtc`) returned 100% nodata for well-covered scenes when the target grid sits at the extreme negative-longitude edge near -180 (UTM zone 1N), silently emptying the *entire* westmost tile column (`col=0`) globally with no exception -- likely the dominant cause of the sharp cutoff immediately east of the antimeridian in the coarsened plot, independent of the MODIS mask reprojection issue in H3. Root cause: odc-geo #208 -- an invalid world-spanning footprint drops the item during item->tile binning (`tyx_bins` empty). Fixed by a one-line upstream odc-stac PR (`wrapdateline=True` in `image_geometry()`); until a release ships it, the Section 10 monkeypatch is the pipeline workaround, and the `col=0` (plus `col=1`/`col=2`) tiles need reprocessing.


## Epilogue (2026-08-13): fixed upstream, in-repo workaround retired

This section closes out everything above. It exists because the code it describes has been
deleted from the repository — this notebook is now the only record of it.

**Timeline**

| when | what |
| --- | --- |
| 2026-07-21 | [odc-stac #281](https://github.com/opendatacube/odc-stac/pull/281) (`GeoBox.footprint(..., wrapdateline=True)`) merged upstream |
| 2026-07-21 → 2026-08-13 | `global_snowmelt_runoff_onset/processing.py` carried a self-verifying guard, `ensure_antimeridian_footprint_fix()`, called at the top of every `get_sentinel1_rtc()` |
| released | the fix shipped in **odc-stac 0.5.3** |
| 2026-08-03 | pin bumped to `odc-stac = ">=0.5.3"` in `pixi.toml` (commit `a000396`), in *both* `[dependencies]` and `[feature.ci.dependencies]` |
| 2026-08-13 | guard deleted — with the pin in place no resolvable environment can lack the fix, so the code was unreachable. Verified in the live env first: odc-stac 0.5.3 installed, functional check returns `True`, monkeypatch never applied |

**What the guard did** (verbatim copy in the next cell). Rather than compare version strings, it
ran a *functional* check — a synthetic UTM-zone-1N item put through the same `safe_geometry` →
`GeoboxTiles.tiles()` path that silently dropped scenes here, reproducing odc-stac's own regression
test (`tests/test_model.py::test_image_geometry_antimeridian`). If the installed odc-stac was
already correct it was a no-op; if not it applied the `wrapdateline=True` monkeypatch and
*re-verified*, raising `RuntimeError` if the bug survived. The escalation to a hard error was
deliberate: this bug's entire danger was that it fails silently with `success=True`, so a
workaround that stopped working had to fail loudly rather than quietly reproduce the original
problem.

**Data impact — no reprocessing was needed.** The v10 store was initialized clean *after* the fix
(2026-08-04) and the fleet ran against fixed odc-stac throughout, so the affected v9 columns 0–2
were superseded rather than repaired. Tile **(10,0)** — v9 (8,0), the tile diagnosed in this
notebook — is the antimeridian entry in the standing validation battery
(`processing/tile_data/test_tiles_v10.txt`) and holds real data in its westernmost columns in v10.

**Still open upstream.** Only the odc-stac binning bug is fixed. [odc-geo #208](https://github.com/opendatacube/odc-geo/issues/208)
is still open, and the `force_over=True` finding worked out earlier in this notebook — a single
coordinate-transform-level fix that addresses [#176](https://github.com/opendatacube/odc-geo/issues/176)
*and* our #208 case, instead of two post-hoc polygon repairs — was never upstreamed. Anyone picking
that up should start from the "possible future odc-geo PR" section above; it has live reproducers
for each member of the bug family.

**Note on running this notebook:** it is deliberately pinned to v9 (it reads the frozen Zarr v2
store and the retired `coarsened/` convention). It is kept as a diagnosis record, not as a runnable
QA notebook — the v10 equivalent of the tile checks here is `processing/3_quality_check_tiles.ipynb`.


In [ ]:
# Retired 2026-08-13 — kept verbatim for posterity, NOT imported by anything.
# This is exactly what global_snowmelt_runoff_onset/processing.py carried between
# 2026-07-21 and 2026-08-13, when odc-stac < 0.5.3 was still installable. It is
# reproduced here so the workaround (and the reasoning in its docstrings) survives
# the deletion. Do not re-add it without first re-checking whether the upstream
# fix is still present: `_odc_stac_handles_antimeridian_footprints()` is the check.

_antimeridian_fix_verified = False


def _odc_stac_handles_antimeridian_footprints() -> bool:
    """
    Functional (not version-based) check for whether ``odc.stac`` correctly computes
    the footprint of a source item whose native UTM zone touches the antimeridian
    (e.g. zone 1N/60N), used internally by ``odc.stac.load()`` to decide which
    destination tile(s) an item overlaps.

    When this is broken, the affected items are silently dropped during tile binning
    with no warning or exception raised — this pipeline's westernmost tile column
    (row col=0, straddling 180°/-180°) came back 100% nodata in every water year despite
    a recorded ``success=True`` status. See this notebook for the full diagnosis, and
    https://github.com/opendatacube/odc-geo/issues/208 /
    https://github.com/opendatacube/odc-stac/pull/281 for the upstream fix
    (``GeoBox.footprint(..., wrapdateline=True)``).

    This reproduces the regression test odc-stac itself ships for the fix
    (``tests/test_model.py::test_image_geometry_antimeridian``), so it correctly
    reports "fixed" whether the fix came from an upstream release or from
    ``_apply_antimeridian_footprint_patch()`` below.
    """
    from affine import Affine
    from odc.geo import CRS
    from odc.geo.geobox import GeoBox, GeoboxTiles
    from odc.stac.testing.stac import b_, mk_parsed_item

    # A UTM zone 1N source geobox straddling the antimeridian (same extent as a real
    # Sentinel-1 RTC scene over this pipeline's Tile(10, 0) -- Tile(8, 0) before the
    # 2026-07-30 grid extension; the ground is the same).
    gbox = GeoBox(
        (25129, 29486), Affine(10.0, 0.0, 281870.0, 0.0, -10.0, 7738430.0), "EPSG:32601"
    )
    item = mk_parsed_item([b_("b1", gbox)])
    footprint = item.safe_geometry(CRS("EPSG:4326"))
    if footprint is None or not footprint.geom.is_valid or footprint.geom.area > 30:
        return False

    # Destination tile grid straddling the antimeridian at the same latitude.
    dst = GeoBox(
        (256, 256),
        Affine(0.00576000576, 0.0, -179.99945999946, 0.0, -0.00576000576, 69.3029493),
        "EPSG:4326",
    )
    tiles = GeoboxTiles(dst, tile_shape=(256, 256))
    return list(tiles.tiles(footprint)) == [(0, 0)]


def _apply_antimeridian_footprint_patch() -> None:
    """
    Monkeypatch ``odc.stac.model.ParsedItem.image_geometry`` to pass
    ``wrapdateline=True`` when reprojecting a source geobox's footprint — byte-equivalent
    to the fix merged upstream in https://github.com/opendatacube/odc-stac/pull/281.
    """
    _osm = odc.stac.model

    def _image_geometry_wrapdateline(self, crs=_osm.Unset(), bands=None):
        if isinstance(crs, _osm.Unset):
            crs = None
        for gbox in self.geoboxes(bands):
            if gbox.crs is not None:
                if crs is None or crs == gbox.crs:
                    return gbox.extent
                return gbox.footprint(crs, wrapdateline=True)
        return None

    _osm.ParsedItem.image_geometry = _image_geometry_wrapdateline


def ensure_antimeridian_footprint_fix() -> None:
    """
    Verify that ``odc.stac.load()`` correctly handles source items whose UTM zone
    touches the antimeridian, applying a local monkeypatch if the installed odc-stac
    doesn't yet ship the upstream fix. Raises ``RuntimeError`` if the bug is still
    present even after patching, since this failure mode is silent (no exception,
    just missing data) and has previously gone unnoticed through a full processing run.

    Cheap (< 10 ms) and idempotent — safe to call at the top of every
    ``get_sentinel1_rtc()`` invocation. Self-cleaning: once odc-stac ships a release
    with the fix and the pin in ``pixi.toml`` is bumped, the functional check passes
    immediately and the monkeypatch is never applied.
    """
    global _antimeridian_fix_verified
    if _antimeridian_fix_verified:
        return

    if _odc_stac_handles_antimeridian_footprints():
        _antimeridian_fix_verified = True
        return

    _apply_antimeridian_footprint_patch()

    if not _odc_stac_handles_antimeridian_footprints():
        raise RuntimeError(
            "odc.stac antimeridian footprint bug is present and the local monkeypatch "
            "(global_snowmelt_runoff_onset.processing._apply_antimeridian_footprint_patch) "
            "did not fix it. Sentinel-1 scenes near 180°/-180° would silently be dropped "
            "from tile binning, producing all-nodata output with no error — refusing to "
            "proceed. odc.stac's internals may have changed since this workaround was "
            "written; see https://github.com/opendatacube/odc-stac/pull/281 and "
            "https://github.com/opendatacube/odc-geo/issues/208."
        )

    warnings.warn(
        "Installed odc-stac does not yet include the antimeridian footprint fix "
        "(https://github.com/opendatacube/odc-stac/pull/281, merged upstream but not "
        "yet in a tagged release) — applied a local monkeypatch for this process. "
        "Once odc-stac releases the fix, bump the pin in pixi.toml and this warning "
        "will stop appearing.",
        stacklevel=2,
    )
    _antimeridian_fix_verified = True
